# RSM-CNBI — versão refatorada com janela singular

Notebook reorganizado para execução sequencial, com correções de robustez e filtro geométrico por **janela singular do CHIM**. A metodologia foi mantida: RSM → payoff → filtragem espectral do CHIM → CNBI → filtros → GMM → GD/entropia → modelos de mistura.


> **Correção anchor-safe:** os vértices do Simplex-Lattice agora reutilizam diretamente os ótimos individuais da payoff; betas internos usam warm start, interpolação das âncoras e retries determinísticos. A execução é auditada para retornar 616/616 pontos válidos.


In [ ]:
# ============================================================
# 0) PRESETS DO NOTEBOOK
# Altere preferencialmente apenas este bloco.
# ============================================================

from pathlib import Path
import os

# -----------------------------
# Arquivo de entrada
# -----------------------------
ARQUIVO = Path("VRF_artigo.xlsx")
ABA = None  # None = primeira aba do Excel

# -----------------------------
# Variáveis do DOE/RSM
# -----------------------------
FACTOR_COLS = ["cs", "f", "md"]
RESPONSE_COLS = ["T","MTTF","WR","Ra","Rt","Kp","ROI","OEE"]


# Sentido original de otimização das respostas
DIRECAO = {
    "T": "max",
    "MTTF":  "max",
    "WR":   "min",
    "Ra":   "min",
    "Rt":  "min",
    "Kp":  "min",
    "ROI":  "max",
    "OEE":  "max",
}
# Mantém compatibilidade com células antigas
direcao = DIRECAO.copy()

# -----------------------------
# Modelo RSM / região experimental
# -----------------------------
RSM_MODELO = "quadratic_full"
RAIO_DOE = 2 ** 0.75  # = 1.6817928..., raio axial exato do CCD rotacional (k=3)


# -----------------------------
# Diagnósticos geométricos / janela singular
# -----------------------------
# A dimensão efetiva d do CHIM global é estimada por análise paralela
# (espectro singular real vs. espectro de CHIMs de ruído puro, obtido por
# Monte Carlo a partir do erro de predição do RSM propagado). Ver célula
# 'DIMENSÃO EFETIVA DO CHIM — ANÁLISE PARALELA COM RUÍDO PROPAGADO DO RSM'.
#   D_CHIM_MANUAL: None = automático (recomendado); um inteiro (ex.: 3) força o valor de d.
D_CHIM_MANUAL = None
N_MC_RUIDO = 2000
SEED_MC_RUIDO = 777
# Modelo de ruído da análise paralela do CHIM:
#   "independente"   — entradas iid N(0, s_ij^2)  (comportamento original);
#   "correlacionado" — linhas ~ N(0, h_i * D^-1 Sigma_res D^-1): mesmas
#                      variâncias marginais, mas incorporando a covariância
#                      RESIDUAL entre respostas (a premissa do trabalho é que
#                      as respostas são correlacionadas; os erros de predição
#                      no mesmo ponto herdam essa estrutura).
# A célula da análise paralela SEMPRE computa os dois espectros e reporta os
# dois valores de d; este preset escolhe qual deles calibra a janela.
RUIDO_MC_MODO = "independente"

USAR_JANELA_SINGULAR = True
# Se False, usa apenas Q_CHIM_CORTE fixo (valor de literatura).
# Se True, usa piso (resíduo geométrico sigma_(d+1)) + teto (sigma_1/sigma_d),
# ambos derivados da SVD do CHIM global.

# -----------------------------
# Otimização dos mínimos individuais
# -----------------------------
N_STARTS_RANDOM_MINIMOS = 2

# -----------------------------
# CNBI / filtros geométricos
# -----------------------------
Q_CHIM_CORTE = 12.5
DELTA_PREVIEW = 0.25
DELTA_POR_K = {
    2: 0.10,
    3: 0.10,
    4: 0.20,
    5: 0.50,
}

TOL_EQ = 1e-5
MAXITER = 1000
BATCH_SIZE_COMBOS = 5
N_JOBS = max(1, (os.cpu_count() or 2) - 2)
BACKEND = "loky"

# -----------------------------
# Clusterização e saídas
# -----------------------------
K_MIN_GMM = 2
K_MAX_GMM_ABSOLUTO = 20
OUTPUT_DIR = Path("resultados_cnbi")
OUTPUT_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)

# Evita oversubscription em paralelismo
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

print("Presets carregados.")
print("Arquivo:", ARQUIVO)
print("Fatores:", FACTOR_COLS)
print("Respostas:", RESPONSE_COLS)
print("Delta por k:", DELTA_POR_K)
print("Saídas:", OUTPUT_DIR.resolve())


In [ ]:
# ============================================================
# 1) IMPORTAÇÕES
# ============================================================

import ast
import itertools
import math
import re
import time
from itertools import combinations, product
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from scipy.optimize import minimize
from scipy.spatial.distance import mahalanobis

try:
    from joblib import Parallel, delayed
    JOBLIB_OK = True
except ImportError:
    JOBLIB_OK = False

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None

# ============================================================
# REGISTRO CENTRAL DO ORÇAMENTO COMPUTACIONAL
# ============================================================
# O orçamento é multidimensional:
#   - avaliações do modelo RSM;
#   - decomposições SVD explícitas;
#   - réplicas Monte Carlo;
#   - subproblemas/combinações processados;
#   - tempo de parede (perf_counter).
#
# Não se convertem SVDs ou réplicas Monte Carlo artificialmente em
# "avaliações equivalentes" do RSM. As moedas são reportadas lado a lado.
from time import perf_counter, process_time

PIPELINE_WALL_T0 = perf_counter()
PIPELINE_CPU_T0 = process_time()
REGISTRO_CUSTO_PIPELINE = {}


def iniciar_medicao_etapa():
    """Marca o início de uma etapa para medir parede e CPU do processo principal."""
    return {
        "wall_t0": perf_counter(),
        "cpu_t0": process_time(),
    }


def registrar_custo_etapa(
    chave,
    ordem,
    categoria,
    timer,
    avaliacoes_rsm=0,
    n_svd=0,
    n_mc=0,
    n_subproblemas=0,
    n_combinacoes=0,
    observacao="",
    **metricas
):
    """
    Registra ou substitui uma etapa no orçamento.

    O uso de uma chave única torna a célula idempotente: se ela for
    reexecutada, o registro anterior da mesma etapa é substituído.
    """
    wall_s = float(perf_counter() - timer["wall_t0"])
    cpu_s = float(process_time() - timer["cpu_t0"])

    registro = {
        "chave": str(chave),
        "ordem": int(ordem),
        "categoria": str(categoria),
        "avaliacoes_rsm": int(avaliacoes_rsm),
        "n_svd": int(n_svd),
        "n_mc": int(n_mc),
        "n_subproblemas": int(n_subproblemas),
        "n_combinacoes": int(n_combinacoes),
        "tempo_parede_s": wall_s,
        # Em etapas paralelas com joblib/loky, esta CPU mede somente o
        # processo principal. O tempo de parede é a medida de referência.
        "tempo_cpu_processo_principal_s": cpu_s,
        "observacao": str(observacao),
    }
    registro.update(metricas)
    REGISTRO_CUSTO_PIPELINE[str(chave)] = registro
    return registro


print("Registro de orçamento e cronômetros inicializados.")


In [ ]:
# ============================================================
# 2) LEITURA DO ARQUIVO
# ============================================================

xls = pd.ExcelFile(ARQUIVO, engine="openpyxl")

print("Abas disponíveis:")
print(xls.sheet_names)

if ABA is None:
    ABA_USADA = xls.sheet_names[0]
else:
    ABA_USADA = ABA

print("Aba usada:", ABA_USADA)

df = pd.read_excel(ARQUIVO, sheet_name=ABA_USADA, engine="openpyxl")
df.columns = df.columns.astype(str).str.strip()

display(df.head())
print(df.columns)


In [ ]:
# ============================================================
# 3) SELEÇÃO DE FATORES E RESPOSTAS
# ============================================================

# Se RESPONSE_COLS estiver vazio ou None, inferir automaticamente.
if RESPONSE_COLS is None or len(RESPONSE_COLS) == 0:
    RESPONSE_COLS = [c for c in df.columns if c not in FACTOR_COLS]

faltando_fatores = [c for c in FACTOR_COLS if c not in df.columns]
faltando_respostas = [c for c in RESPONSE_COLS if c not in df.columns]

if faltando_fatores:
    raise ValueError(f"Fatores ausentes no df: {faltando_fatores}")
if faltando_respostas:
    raise ValueError(f"Respostas ausentes no df: {faltando_respostas}")

df = df[FACTOR_COLS + RESPONSE_COLS].copy()

print("Fatores:", FACTOR_COLS)
print("Respostas:", RESPONSE_COLS)


In [ ]:
# ------------------------------------------------------------
# 2) MATRIZES DOE: FATORIAL, AXIAL, CENTRAL E CCD
# ------------------------------------------------------------
def gerar_fatorial_completo(k, nomes=None):
    """Gera matriz fatorial 2^k em níveis -1 e +1."""
    nomes = nomes or [f"x{i+1}" for i in range(k)]
    mat = np.array(list(product([-1, 1], repeat=k)), dtype=float)
    return pd.DataFrame(mat, columns=nomes)


def gerar_axial(k, alpha=None, nomes=None):
    """Gera pontos axiais de um CCD."""
    nomes = nomes or [f"x{i+1}" for i in range(k)]
    if alpha is None:
        alpha = (2 ** k) ** 0.25  # alpha rotacional
    pontos = []
    for i in range(k):
        p1 = np.zeros(k)
        p2 = np.zeros(k)
        p1[i] = -alpha
        p2[i] = alpha
        pontos.append(p1)
        pontos.append(p2)
    return pd.DataFrame(pontos, columns=nomes)


def gerar_centrais(k, n_center=1, nomes=None):
    """Gera pontos centrais."""
    nomes = nomes or [f"x{i+1}" for i in range(k)]
    return pd.DataFrame(np.zeros((n_center, k)), columns=nomes)


def gerar_ccd(k, alpha=None, n_center=1, nomes=None):
    """Gera CCD = fatorial + axial + central."""
    nomes = nomes or [f"x{i+1}" for i in range(k)]
    fat = gerar_fatorial_completo(k, nomes)
    ax = gerar_axial(k, alpha, nomes)
    cen = gerar_centrais(k, n_center, nomes)
    ccd = pd.concat([fat, ax, cen], ignore_index=True)
    return ccd


def classificar_ponto_ccd(row, tol=1e-9):
    """Classifica linha do DOE real: fatorial, axial, central ou outro."""
    x = np.asarray(row, dtype=float)
    nz = np.sum(np.abs(x) > tol)

    if np.all(np.abs(x) <= tol):
        return "central"

    if nz == 1:
        return "axial"

    if np.all(np.isclose(np.abs(x), 1.0, atol=tol)):
        return "fatorial"

    return "outro"


k = len(FACTOR_COLS)
alpha_detectado = np.nanmax(np.abs(df[FACTOR_COLS].values))

matriz_fatorial_teorica = gerar_fatorial_completo(k, FACTOR_COLS)
matriz_axial_teorica = gerar_axial(k, alpha_detectado, FACTOR_COLS)
matriz_central_teorica = gerar_centrais(
    k,
    n_center=(df[FACTOR_COLS].abs().sum(axis=1).eq(0).sum()),
    nomes=FACTOR_COLS
)
matriz_ccd_teorica = pd.concat(
    [matriz_fatorial_teorica, matriz_axial_teorica, matriz_central_teorica],
    ignore_index=True
)

df["tipo_ponto"] = df[FACTOR_COLS].apply(classificar_ponto_ccd, axis=1)

print("\nResumo dos pontos do DOE:")
display(df["tipo_ponto"].value_counts())

print("\nMatriz fatorial teórica:")
display(matriz_fatorial_teorica)

print("\nMatriz axial teórica:")
display(matriz_axial_teorica)

print("\nMatriz central teórica:")
display(matriz_central_teorica.head())

In [ ]:
# ------------------------------------------------------------
# 3) MATRIZ DO MODELO RSM
# Modelos:
#   linear           = 1 + xi
#   2FI              = 1 + xi + xixj
#   pure_quadratic   = 1 + xi + xi²
#   quadratic_full   = 1 + xi + xi² + xixj
# ------------------------------------------------------------
def build_design_matrix(df_x, factor_cols, modelo="quadratic_full"):
    """
    Monta a matriz X do modelo em variáveis codificadas.
    """
    Xraw = df_x[factor_cols].astype(float).copy()
    out = pd.DataFrame(index=df_x.index)
    out["Intercepto"] = 1.0

    # Termos lineares
    for c in factor_cols:
        out[c] = Xraw[c]

    # Termos quadráticos puros
    if modelo in ["pure_quadratic", "quadratic_full"]:
        for c in factor_cols:
            out[f"{c}^2"] = Xraw[c] ** 2

    # Termos de interação
    if modelo in ["2FI", "quadratic_full"]:
        for a, b in combinations(factor_cols, 2):
            out[f"{a}:{b}"] = Xraw[a] * Xraw[b]

    return out


MODELOS = ["linear", "2FI", "pure_quadratic", "quadratic_full"]

matrizes_X = {
    modelo: build_design_matrix(df, FACTOR_COLS, modelo)
    for modelo in MODELOS
}

for nome, Xmat in matrizes_X.items():
    print(f"\nMatriz X - modelo {nome}: shape = {Xmat.shape}")
    display(Xmat.head())


In [ ]:
# ------------------------------------------------------------
# 4) AJUSTE OLS MATRICIAL
# Fórmulas:
#   beta = (X'X)^(-1) X'Y
#   Yhat = X beta
#   e = Y - Yhat
#   H = X (X'X)^(-1) X'
#   e_PRESS = e / (1 - h_ii)
#   PRESS = soma(e_PRESS²)
# ------------------------------------------------------------
def ols_matricial(X_df, Y_df):
    X = X_df.values.astype(float)
    Y = Y_df.values.astype(float)

    n, p = X.shape
    m = Y.shape[1]

    XtX = X.T @ X
    XtY = X.T @ Y

    try:
        XtX_inv = np.linalg.inv(XtX)
        inverse_used = "inv"
    except np.linalg.LinAlgError:
        XtX_inv = np.linalg.pinv(XtX)
        inverse_used = "pinv"

    B = XtX_inv @ XtY
    Yhat = X @ B
    E = Y - Yhat

    H = X @ XtX_inv @ X.T
    h = np.diag(H).reshape(-1, 1)

    denom_press = 1 - h
    denom_press_safe = np.where(np.abs(denom_press) > 1e-10, denom_press, np.nan)
    press_residuals = E / denom_press_safe
    PRESS = np.nansum(press_residuals ** 2, axis=0)
    if np.any(~np.isfinite(press_residuals)):
        print('Aviso: PRESS contém pontos com h_ii ≈ 1; resíduos PRESS desses pontos foram ignorados com nan.')

    SSE = np.sum(E ** 2, axis=0)
    Ybar = np.mean(Y, axis=0)
    SST = np.sum((Y - Ybar) ** 2, axis=0)
    SSR = SST - SSE

    df_reg = p - 1
    df_err = n - p
    df_total = n - 1

    MSR = SSR / df_reg
    MSE = SSE / df_err

    F_reg = MSR / MSE
    p_reg = 1 - stats.f.cdf(F_reg, df_reg, df_err)

    R2 = 1 - SSE / SST
    R2_adj = 1 - (SSE / df_err) / (SST / df_total)
    R2_PRESS = 1 - PRESS / SST

    coef_df = pd.DataFrame(
        B,
        index=X_df.columns,
        columns=Y_df.columns
    )

    yhat_df = pd.DataFrame(
        Yhat,
        index=Y_df.index,
        columns=[f"{c}_pred" for c in Y_df.columns]
    )

    resid_df = pd.DataFrame(
        E,
        index=Y_df.index,
        columns=[f"{c}_resid" for c in Y_df.columns]
    )

    press_resid_df = pd.DataFrame(
        press_residuals,
        index=Y_df.index,
        columns=[f"{c}_PRESS_resid" for c in Y_df.columns]
    )

    metricas_df = pd.DataFrame({
        "Resposta": Y_df.columns,
        "n": n,
        "p": p,
        "gl_reg": df_reg,
        "gl_erro": df_err,
        "SST": SST,
        "SSR": SSR,
        "SSE": SSE,
        "MSR": MSR,
        "MSE": MSE,
        "F_reg": F_reg,
        "p_reg": p_reg,
        "R2": R2,
        "R2_adj": R2_adj,
        "PRESS": PRESS,
        "R2_PRESS": R2_PRESS,
        "inversa": inverse_used
    })

    anova_df = pd.concat([
        pd.DataFrame({
            "Resposta": Y_df.columns,
            "Fonte": "Regressão",
            "GL": df_reg,
            "SQ": SSR,
            "MQ": MSR,
            "F": F_reg,
            "p": p_reg
        }),
        pd.DataFrame({
            "Resposta": Y_df.columns,
            "Fonte": "Erro",
            "GL": df_err,
            "SQ": SSE,
            "MQ": MSE,
            "F": np.nan,
            "p": np.nan
        }),
        pd.DataFrame({
            "Resposta": Y_df.columns,
            "Fonte": "Total",
            "GL": df_total,
            "SQ": SST,
            "MQ": np.nan,
            "F": np.nan,
            "p": np.nan
        })
    ], ignore_index=True)

    return {
        "X": X_df,
        "Y": Y_df,
        "XtX": pd.DataFrame(XtX, index=X_df.columns, columns=X_df.columns),
        "XtX_inv": pd.DataFrame(XtX_inv, index=X_df.columns, columns=X_df.columns),
        "XtY": pd.DataFrame(XtY, index=X_df.columns, columns=Y_df.columns),
        "B_coeficientes": coef_df,
        "Yhat": yhat_df,
        "residuos": resid_df,
        "H": pd.DataFrame(H, index=Y_df.index, columns=Y_df.index),
        "h_ii": pd.DataFrame(h, index=Y_df.index, columns=["h_ii"]),
        "residuos_PRESS": press_resid_df,
        "metricas": metricas_df,
        "ANOVA": anova_df
    }


Y_df = df[RESPONSE_COLS].astype(float)

resultados = {}
for modelo in MODELOS:
    resultados[modelo] = ols_matricial(matrizes_X[modelo], Y_df)

In [ ]:
# ------------------------------------------------------------
# 5) TABELA FINAL DE R² AJUSTADO E PRESS
# ------------------------------------------------------------
metricas_todos_modelos = []

for modelo, res in resultados.items():
    tmp = res["metricas"].copy()
    tmp.insert(0, "Modelo", modelo)
    metricas_todos_modelos.append(tmp)

metricas_todos_modelos = pd.concat(metricas_todos_modelos, ignore_index=True)

cols_show = [
    "Modelo", "Resposta", "n", "p",
    "R2", "R2_adj", "PRESS", "R2_PRESS",
    "SSE", "MSE", "F_reg", "p_reg"
]

display(
    metricas_todos_modelos[cols_show]
    .sort_values(["Resposta", "Modelo"])
    .style.format({
        "R2": "{:.5f}",
        "R2_adj": "{:.5f}",
        "PRESS": "{:.6g}",
        "R2_PRESS": "{:.5f}",
        "SSE": "{:.6g}",
        "MSE": "{:.6g}",
        "F_reg": "{:.4f}",
        "p_reg": "{:.4g}"
    })
)

In [ ]:
# ------------------------------------------------------------
# 6) LACK OF FIT E ERRO PURO
# Útil quando há pontos repetidos, especialmente ponto central.
# ------------------------------------------------------------
def lack_of_fit(df_original, factor_cols, Y_df, resultado_modelo):
    X_df = resultado_modelo["X"]
    p = X_df.shape[1]
    n = X_df.shape[0]

    grupos = df_original.groupby(factor_cols, dropna=False).groups
    g = len(grupos)

    out = []

    for resposta in Y_df.columns:
        y = Y_df[resposta].values.astype(float)
        yhat = resultado_modelo["Yhat"][f"{resposta}_pred"].values.astype(float)
        resid = y - yhat

        SSE = np.sum(resid ** 2)

        SS_pe = 0.0
        df_pe = 0

        for _, idx in grupos.items():
            idx = list(idx)
            vals = y[idx]
            if len(vals) > 1:
                SS_pe += np.sum((vals - np.mean(vals)) ** 2)
                df_pe += len(vals) - 1

        SS_lof = max(SSE - SS_pe, 0)
        df_lof = g - p

        if df_pe > 0 and df_lof > 0:
            MS_pe = SS_pe / df_pe
            MS_lof = SS_lof / df_lof
            F_lof = MS_lof / MS_pe if MS_pe > 0 else np.nan
            p_lof = 1 - stats.f.cdf(F_lof, df_lof, df_pe)
        else:
            MS_pe = np.nan
            MS_lof = np.nan
            F_lof = np.nan
            p_lof = np.nan

        out.append({
            "Resposta": resposta,
            "n": n,
            "p": p,
            "grupos_unicos": g,
            "SS_erro_total": SSE,
            "SS_erro_puro": SS_pe,
            "SS_lack_of_fit": SS_lof,
            "GL_erro_puro": df_pe,
            "GL_lack_of_fit": df_lof,
            "MS_erro_puro": MS_pe,
            "MS_lack_of_fit": MS_lof,
            "F_lack_of_fit": F_lof,
            "p_lack_of_fit": p_lof
        })

    return pd.DataFrame(out)


lof_quad = lack_of_fit(df, FACTOR_COLS, Y_df, resultados["quadratic_full"])

print("\nLack of fit - modelo quadrático completo:")
display(
    lof_quad.style.format({
        "SS_erro_total": "{:.6g}",
        "SS_erro_puro": "{:.6g}",
        "SS_lack_of_fit": "{:.6g}",
        "MS_erro_puro": "{:.6g}",
        "MS_lack_of_fit": "{:.6g}",
        "F_lack_of_fit": "{:.4f}",
        "p_lack_of_fit": "{:.4g}"
    })
)

In [ ]:
# ------------------------------------------------------------
# 7) EQUAÇÕES RSM AJUSTADAS
# ------------------------------------------------------------
def imprimir_equacoes(coef_df, casas=5):
    for resposta in coef_df.columns:
        termos = []
        for termo, beta in coef_df[resposta].items():
            if termo == "Intercepto":
                termos.append(f"{beta:.{casas}f}")
            else:
                sinal = "+" if beta >= 0 else "-"
                termos.append(f" {sinal} {abs(beta):.{casas}f}*{termo}")
        eq = f"{resposta} = " + "".join(termos)
        print(eq)


print("\nEquações - modelo quadrático completo:")
imprimir_equacoes(resultados["quadratic_full"]["B_coeficientes"])

In [ ]:
# ============================================================
# PREDICTED VS. ACTUAL PLOTS — 8 RSM RESPONSES
# Layout: 2 rows × 4 columns
# ============================================================

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# 1) Configuration
# ------------------------------------------------------------
MODEL_NAME = RSM_MODELO
RESPONSES = list(RESPONSE_COLS)

if len(RESPONSES) != 8:
    raise ValueError(
        f"This layout requires exactly 8 responses, "
        f"but {len(RESPONSES)} were found: {RESPONSES}"
    )

if MODEL_NAME not in resultados:
    raise KeyError(
        f"Model '{MODEL_NAME}' was not found in 'resultados'. "
        f"Available models: {list(resultados.keys())}"
    )


# ------------------------------------------------------------
# 2) Actual values, predicted values and metrics
# ------------------------------------------------------------
actual_df = resultados[MODEL_NAME]["Y"].copy()
predicted_df = resultados[MODEL_NAME]["Yhat"].copy()

metrics_df = (
    resultados[MODEL_NAME]["metricas"]
    .copy()
    .set_index("Resposta")
)

missing_actual = [
    response
    for response in RESPONSES
    if response not in actual_df.columns
]

missing_predicted = [
    f"{response}_pred"
    for response in RESPONSES
    if f"{response}_pred" not in predicted_df.columns
]

if missing_actual:
    raise KeyError(
        f"Actual response columns not found: {missing_actual}"
    )

if missing_predicted:
    raise KeyError(
        f"Predicted response columns not found: {missing_predicted}"
    )


# ------------------------------------------------------------
# 3) Visual style
# ------------------------------------------------------------
sns.set_theme(
    style="white",
    context="notebook",
    font_scale=1.0
)

plt.rcParams.update({
    "font.family": "sans-serif",
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "axes.linewidth": 1.0,
    "xtick.direction": "out",
    "ytick.direction": "out"
})


# ------------------------------------------------------------
# 4) Create the 2 × 4 figure
# ------------------------------------------------------------
fig, axes = plt.subplots(
    nrows=2,
    ncols=4,
    figsize=(18, 8.5)
)

axes = axes.flatten()


# ------------------------------------------------------------
# 5) Predicted versus actual plots
# ------------------------------------------------------------
for panel_index, response in enumerate(RESPONSES):

    ax = axes[panel_index]

    row = panel_index // 4
    column = panel_index % 4

    predicted_column = f"{response}_pred"

    plot_data = pd.DataFrame({
        "Actual": pd.to_numeric(
            actual_df[response],
            errors="coerce"
        ),
        "Predicted": pd.to_numeric(
            predicted_df[predicted_column],
            errors="coerce"
        )
    }).dropna()

    if plot_data.empty:
        ax.text(
            0.5,
            0.5,
            "No valid data",
            ha="center",
            va="center",
            transform=ax.transAxes,
            fontsize=11,
            weight="bold"
        )

        ax.set_axis_off()
        continue

    y_actual = plot_data["Actual"].to_numpy(dtype=float)
    y_predicted = plot_data["Predicted"].to_numpy(dtype=float)

    # --------------------------------------------------------
    # Common limits for both axes
    # --------------------------------------------------------
    data_min = np.nanmin(
        np.concatenate([y_actual, y_predicted])
    )

    data_max = np.nanmax(
        np.concatenate([y_actual, y_predicted])
    )

    data_range = data_max - data_min

    if not np.isfinite(data_range) or data_range == 0:
        padding = max(abs(data_min) * 0.05, 0.05)
    else:
        padding = 0.07 * data_range

    lower_limit = data_min - padding
    upper_limit = data_max + padding

    # --------------------------------------------------------
    # Data points
    # --------------------------------------------------------
    ax.scatter(
        y_actual,
        y_predicted,
        s=52,
        alpha=0.85,
        edgecolors="white",
        linewidths=0.6,
        zorder=3
    )

    # Perfect-prediction line
    ax.plot(
        [lower_limit, upper_limit],
        [lower_limit, upper_limit],
        linestyle="--",
        linewidth=1.4,
        color="black",
        zorder=2
    )

    ax.set_xlim(lower_limit, upper_limit)
    ax.set_ylim(lower_limit, upper_limit)

    ax.set_aspect(
        "equal",
        adjustable="box"
    )

    # --------------------------------------------------------
    # Panel title
    # --------------------------------------------------------
    panel_letter = chr(ord("a") + panel_index)

    ax.set_title(
        f"({panel_letter}) {response}",
        fontsize=12,
        weight="bold",
        pad=7
    )

    # --------------------------------------------------------
    # Axis labels
    #
    # Predicted value: only on the first plot of each row
    # Actual value: only on the bottom row
    # --------------------------------------------------------
    if column == 0:
        ax.set_ylabel(
            "Predicted value",
            fontsize=11,
            weight="bold"
        )
    else:
        ax.set_ylabel("")

    if row == 1:
        ax.set_xlabel(
            "Actual value",
            fontsize=11,
            weight="bold"
        )
    else:
        ax.set_xlabel("")

    # --------------------------------------------------------
    # Adjusted R² only
    # --------------------------------------------------------
    if response in metrics_df.index:

        r2_adjusted = metrics_df.loc[response, "R2_adj"]

        ax.text(
            0.05,
            0.95,
            rf"Adjusted $R^2$ = {r2_adjusted:.3f}",
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=9.5,
            weight="bold",
            bbox={
                "boxstyle": "round,pad=0.25",
                "facecolor": "white",
                "edgecolor": "0.75",
                "alpha": 0.90
            },
            zorder=4
        )

    # --------------------------------------------------------
    # Grid and borders
    # --------------------------------------------------------
    ax.grid(
        True,
        linestyle="--",
        linewidth=0.5,
        alpha=0.30,
        zorder=1
    )

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.0)

    ax.tick_params(
        axis="both",
        labelsize=8.5
    )


# ------------------------------------------------------------
# 6) Spacing
# No overall figure title and no legend
# ------------------------------------------------------------
fig.subplots_adjust(
    left=0.065,
    right=0.99,
    bottom=0.10,
    top=0.95,
    wspace=0.22,
    hspace=0.25
)

plt.show()

In [ ]:
# ============================================================
# CONTOUR PLOTS — 8 RESPONSES IN A SINGLE 3 × 3 FIGURE
# Fixed factor: cs = 0.0
# Axes: x = md, y = f
# Last row: two centered panels
# Colormap: nipy_spectral
# ============================================================

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize


# ------------------------------------------------------------
# 1) Configuration
# ------------------------------------------------------------
MODEL_NAME = RSM_MODELO
RESPONSES = list(RESPONSE_COLS)

if len(RESPONSES) != 8:
    raise ValueError(
        f"This layout requires exactly 8 responses, "
        f"but {len(RESPONSES)} were found: {RESPONSES}"
    )

if MODEL_NAME not in resultados:
    raise KeyError(
        f"Model '{MODEL_NAME}' was not found in 'resultados'. "
        f"Available models: {list(resultados.keys())}"
    )

# Fixed value for cs
CS_FIXED = 0.0

# Grid settings
GRID_N = 160
N_LEVELS = 20

# Experimental limits
LIM_VAL = (
    alpha_detectado
    if "alpha_detectado" in globals()
    else RAIO_DOE
)

LIM = (-LIM_VAL, LIM_VAL)

# Show experimental points projected on this slice?
SHOW_POINTS = True

# Tolerance to show DOE points close to cs = 0
CS_TOL = 0.20

# Colormap
CMAP = "nipy_spectral"


# ------------------------------------------------------------
# 2) Helper: prediction from fitted model
# ------------------------------------------------------------
def predict_from_fit(
    fit,
    df_x_new,
    factor_cols,
    modelo="quadratic_full"
):
    Xnew = build_design_matrix(
        df_x_new,
        factor_cols,
        modelo
    )

    B = fit["B_coeficientes"]

    return Xnew.values @ B.values


# ------------------------------------------------------------
# 3) Visual style
# ------------------------------------------------------------
sns.set_theme(
    style="white",
    context="talk",
    font_scale=1.0
)

plt.rcParams.update({
    "font.family": "sans-serif",

    # Titles
    "axes.titlesize": 16,
    "axes.titleweight": "bold",

    # Axis labels
    "axes.labelsize": 15,
    "axes.labelweight": "bold",

    # Tick labels
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,

    # Lines and ticks
    "axes.linewidth": 1.1,
    "xtick.direction": "out",
    "ytick.direction": "out"
})


# ------------------------------------------------------------
# 4) Prepare fit and DOE points
# ------------------------------------------------------------
fit = resultados[MODEL_NAME]

if "df" not in globals():
    raise NameError(
        "The dataframe 'df' was not found in memory."
    )

if not all(c in df.columns for c in FACTOR_COLS):
    raise ValueError(
        f"The dataframe 'df' must contain the factor columns "
        f"{FACTOR_COLS}."
    )

df_points = df.copy()


# ------------------------------------------------------------
# 5) Build common grid for the slice cs = 0.0
# ------------------------------------------------------------
md_vals = np.linspace(
    LIM[0],
    LIM[1],
    GRID_N
)

f_vals = np.linspace(
    LIM[0],
    LIM[1],
    GRID_N
)

MD, F = np.meshgrid(
    md_vals,
    f_vals
)

grid_df = pd.DataFrame({
    "cs": np.full(
        MD.size,
        CS_FIXED,
        dtype=float
    ),
    "f": F.ravel(),
    "md": MD.ravel()
})


# Predict all responses at once
Y_grid = predict_from_fit(
    fit=fit,
    df_x_new=grid_df,
    factor_cols=FACTOR_COLS,
    modelo=MODEL_NAME
)

Y_grid_df = pd.DataFrame(
    Y_grid,
    columns=RESPONSES
)


# ------------------------------------------------------------
# 6) Spherical DOE mask
# ------------------------------------------------------------
mask_inside = (
    grid_df["cs"].to_numpy(dtype=float) ** 2
    + grid_df["f"].to_numpy(dtype=float) ** 2
    + grid_df["md"].to_numpy(dtype=float) ** 2
) <= (RAIO_DOE ** 2 + 1e-12)

mask_inside = mask_inside.reshape(MD.shape)


# Boundary radius of the circular slice
slice_radius_sq = max(
    RAIO_DOE ** 2 - CS_FIXED ** 2,
    0.0
)

slice_radius = np.sqrt(slice_radius_sq)


# ------------------------------------------------------------
# 7) Identify experimental points near the slice
# ------------------------------------------------------------
if SHOW_POINTS:
    near_slice = np.isclose(
        df_points["cs"].to_numpy(dtype=float),
        CS_FIXED,
        atol=CS_TOL
    )
else:
    near_slice = np.zeros(
        len(df_points),
        dtype=bool
    )


# ------------------------------------------------------------
# 8) Create figure using GridSpec
#
# Six internal columns are used so that the two panels in the
# last row can be shifted by half a panel and centered.
#
# Layout:
#
# (a)       (b)       (c)
# (d)       (e)       (f)
#      (g)       (h)
# ------------------------------------------------------------
fig = plt.figure(
    figsize=(16, 12.5)
)

gs = fig.add_gridspec(
    nrows=3,
    ncols=6,
    left=0.065,
    right=0.885,
    bottom=0.075,
    top=0.965,
    wspace=0.45,
    hspace=0.42
)


# Each regular panel occupies two GridSpec columns.
# The last two panels are shifted by one GridSpec column.
panel_positions = [
    # First row
    (0, slice(0, 2)),
    (0, slice(2, 4)),
    (0, slice(4, 6)),

    # Second row
    (1, slice(0, 2)),
    (1, slice(2, 4)),
    (1, slice(4, 6)),

    # Third row, centered
    (2, slice(1, 3)),
    (2, slice(3, 5))
]


axes = [
    fig.add_subplot(gs[row, columns])
    for row, columns in panel_positions
]


# ------------------------------------------------------------
# 9) Draw each contour plot
# ------------------------------------------------------------
for panel_index, response in enumerate(RESPONSES):

    ax = axes[panel_index]

    Z = (
        Y_grid_df[response]
        .to_numpy(dtype=float)
        .reshape(MD.shape)
    )

    # Hide grid points outside the spherical experimental region
    Z = np.where(
        mask_inside,
        Z,
        np.nan
    )


    # --------------------------------------------------------
    # Filled contours
    # --------------------------------------------------------
    cf = ax.contourf(
        MD,
        F,
        Z,
        levels=N_LEVELS,
        cmap=CMAP
    )


    # --------------------------------------------------------
    # Contour lines
    # --------------------------------------------------------
    cl = ax.contour(
        MD,
        F,
        Z,
        levels=N_LEVELS,
        colors="black",
        linewidths=0.55,
        alpha=0.60
    )


    # Label every second contour line
    ax.clabel(
        cl,
        levels=cl.levels[::2],
        inline=True,
        inline_spacing=3,
        fontsize=9.5,
        fmt="%.2f"
    )


    # --------------------------------------------------------
    # Circular boundary of the DOE slice
    # --------------------------------------------------------
    if slice_radius > 0:

        theta = np.linspace(
            0,
            2 * np.pi,
            600
        )

        x_circle = (
            slice_radius
            * np.cos(theta)
        )

        y_circle = (
            slice_radius
            * np.sin(theta)
        )

        ax.plot(
            x_circle,
            y_circle,
            color="black",
            linewidth=1.3,
            linestyle="--",
            zorder=3
        )


    # --------------------------------------------------------
    # Experimental points near cs = 0
    # --------------------------------------------------------
    if SHOW_POINTS and np.any(near_slice):

        ax.scatter(
            df_points.loc[near_slice, "md"],
            df_points.loc[near_slice, "f"],
            s=42,
            c="white",
            edgecolors="black",
            linewidths=0.9,
            alpha=0.95,
            zorder=4
        )


    # --------------------------------------------------------
    # Panel title
    # --------------------------------------------------------
    panel_letter = chr(
        ord("a") + panel_index
    )

    ax.set_title(
        f"({panel_letter}) {response}",
        fontsize=16,
        weight="bold",
        pad=10
    )


    # --------------------------------------------------------
    # Axis limits and proportions
    # --------------------------------------------------------
    ax.set_xlim(LIM)
    ax.set_ylim(LIM)

    ax.set_aspect(
        "equal",
        adjustable="box"
    )


    # --------------------------------------------------------
    # Axis labels
    # --------------------------------------------------------

    # Y-axis label on first panel of each row
    if panel_index in [0, 3, 6]:
        ax.set_ylabel(
            "f",
            fontsize=15,
            weight="bold",
            labelpad=7
        )
    else:
        ax.set_ylabel("")


    # X-axis label on the final row
    if panel_index in [6, 7]:
        ax.set_xlabel(
            "md",
            fontsize=15,
            weight="bold",
            labelpad=7
        )
    else:
        ax.set_xlabel("")


    # --------------------------------------------------------
    # Grid
    # --------------------------------------------------------
    ax.grid(
        True,
        linestyle="--",
        linewidth=0.5,
        alpha=0.25
    )


    # --------------------------------------------------------
    # Panel borders
    # --------------------------------------------------------
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.1)


    # --------------------------------------------------------
    # Tick formatting
    # --------------------------------------------------------
    ax.tick_params(
        axis="both",
        which="major",
        labelsize=12,
        width=1.0,
        length=4
    )


# ------------------------------------------------------------
# 10) Global qualitative colorbar
#
# Because each response has a different physical scale, this
# colorbar represents only the relative direction:
# lower response level at the bottom and higher at the top.
# ------------------------------------------------------------
norm_cb = Normalize(
    vmin=-1,
    vmax=1
)

sm = ScalarMappable(
    norm=norm_cb,
    cmap=CMAP
)

sm.set_array([])


# Dedicated axis for the colorbar
cax = fig.add_axes([
    0.16,  # horizontal position
    0.0,   # vertical position
    0.6,  # width
    0.02    # height
])


cbar = fig.colorbar(
    sm,
    cax=cax,
    orientation="horizontal"
)

cbar.set_ticks([
    -1,
    1
])

cbar.set_ticklabels([
    "−",
    "+"
])

cbar.ax.tick_params(
    labelsize=17,
    width=1.1,
    length=5
)

cbar.set_label(
    "Response level",
    fontsize=15,
    weight="bold",
    labelpad=11
)


# ------------------------------------------------------------
# 11) Show figure
# ------------------------------------------------------------
plt.show()

In [ ]:
# ------------------------------------------------------------
# 9) EXPORTAR TODAS AS MATRIZES PARA EXCEL
# ------------------------------------------------------------
def safe_sheet_name(name):
    invalid = ["\\", "/", "*", "[", "]", ":", "?"]
    for ch in invalid:
        name = name.replace(ch, "_")
    return name[:31]


EXPORTAR = True

if EXPORTAR:
    saida = "matrizes_DOE_RSM_OLS_PRESSvrf.xlsx"

    with pd.ExcelWriter(saida, engine="openpyxl") as writer:
        df.to_excel(writer, sheet_name="Dados", index=False)

        matriz_fatorial_teorica.to_excel(writer, sheet_name="DOE_fatorial", index=False)
        matriz_axial_teorica.to_excel(writer, sheet_name="DOE_axial", index=False)
        matriz_central_teorica.to_excel(writer, sheet_name="DOE_central", index=False)
        matriz_ccd_teorica.to_excel(writer, sheet_name="DOE_CCD_teorico", index=False)

        metricas_todos_modelos.to_excel(writer, sheet_name="Metricas_modelos", index=False)
        lof_quad.to_excel(writer, sheet_name="Lack_of_fit_quad", index=False)

        for modelo, res in resultados.items():
            prefix = safe_sheet_name(modelo)

            res["X"].to_excel(writer, sheet_name=safe_sheet_name(f"{prefix}_X"))
            res["XtX"].to_excel(writer, sheet_name=safe_sheet_name(f"{prefix}_XtX"))
            res["XtX_inv"].to_excel(writer, sheet_name=safe_sheet_name(f"{prefix}_XtX_inv"))
            res["XtY"].to_excel(writer, sheet_name=safe_sheet_name(f"{prefix}_XtY"))
            res["B_coeficientes"].to_excel(writer, sheet_name=safe_sheet_name(f"{prefix}_Beta"))
            res["Yhat"].to_excel(writer, sheet_name=safe_sheet_name(f"{prefix}_Yhat"))
            res["residuos"].to_excel(writer, sheet_name=safe_sheet_name(f"{prefix}_Residuos"))
            res["h_ii"].to_excel(writer, sheet_name=safe_sheet_name(f"{prefix}_hii"))
            res["residuos_PRESS"].to_excel(writer, sheet_name=safe_sheet_name(f"{prefix}_PRESSres"))
            res["ANOVA"].to_excel(writer, sheet_name=safe_sheet_name(f"{prefix}_ANOVA"), index=False)

            # H pode ser grande; aqui ainda é pequeno.
            res["H"].to_excel(writer, sheet_name=safe_sheet_name(f"{prefix}_H"))

    print(f"Arquivo exportado: {saida}")

In [ ]:
# ============================================================
# CRIAR Y_min: RESPOSTAS COM TODAS EM SENTIDO DE MINIMIZAÇÃO
# ============================================================


# Conferir se as colunas existem no df
faltando = [c for c in RESPONSE_COLS if c not in df.columns]
if faltando:
    raise ValueError(f"Estas colunas não existem no df: {faltando}")

# Cria matriz original das respostas
Y_original = df[RESPONSE_COLS].copy()

# Cria matriz convertida para minimização
Y_min = Y_original.copy()

for col in RESPONSE_COLS:
    if DIRECAO[col] == "max":
        Y_min[col] = -Y_min[col]
    elif DIRECAO[col] == "min":
        Y_min[col] = Y_min[col]
    else:
        raise ValueError(f"Direção inválida para {col}: {DIRECAO[col]}")

print("Matriz original:")
display(Y_original.head())

print("Matriz convertida para minimização:")
display(Y_min.head())

In [ ]:
# ============================================================
# HEATMAP DE CORRELAÇÃO DE SPEARMAN
# MATRIZ DE RESPOSTAS JÁ CONVERTIDA PARA MINIMIZAÇÃO
# ============================================================

import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# ------------------------------------------------------------
# 1) Escolher a matriz para correlação
# ------------------------------------------------------------
# Use Y_min se você quer a correlação após inverter o sinal dos objetivos Max.
# Ou use df[RESPONSE_COLS] se quiser a correlação nas respostas físicas originais.

df_corr = Y_min.copy()

# Garante apenas colunas numéricas
df_num = df_corr.select_dtypes(include=[np.number]).copy()

# Remove colunas constantes, se existirem
df_num = df_num.loc[:, df_num.nunique(dropna=True) > 1]

col_names = df_num.columns

# ------------------------------------------------------------
# 2) Correlação de Spearman e p-values
# ------------------------------------------------------------
rho, p_values = spearmanr(df_num, nan_policy="omit")

rho_df = pd.DataFrame(rho, index=col_names, columns=col_names)
pval_df = pd.DataFrame(p_values, index=col_names, columns=col_names)

# ------------------------------------------------------------
# 3) Matriz de anotação
# ------------------------------------------------------------
annot_matrix = (
    rho_df.map(lambda x: f"{x:.2f}") 
    + "\n" +
    pval_df.map(lambda x: f"(p={x:.3f})")
)

# ------------------------------------------------------------
# 4) Máscara para ocultar triângulo superior
# ------------------------------------------------------------
mask = np.triu(np.ones_like(rho_df, dtype=bool))

# ------------------------------------------------------------
# 5) Plotagem
# ------------------------------------------------------------
plt.figure(figsize=(15, 12))

sns.heatmap(
    rho_df,
    mask=mask,
    cmap="coolwarm_r",
    center=0,
    vmin=-1,
    vmax=1,
    annot=annot_matrix,
    fmt="",
    annot_kws={"size": 12, "weight": "bold"},
    cbar_kws={"ticks": [-1, 0, 1], "shrink": 0.8},
    linewidths=0.5,
    linecolor="white",
    square=True
)


plt.xticks(size=13, rotation=45, ha="right")
plt.yticks(size=13, rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# matriz somente com respostas
Y = df[RESPONSE_COLS].copy()

# garante valores numéricos
Y = Y.apply(pd.to_numeric, errors="coerce")

# remove linhas com resposta faltante
Y_valid = Y.dropna()

# média amostral das respostas
media_respostas = Y_valid.mean()

# variância amostral de cada resposta
variancia_amostral = Y_valid.var(ddof=1)

# matriz de covariância amostral entre respostas
matriz_covariancia = Y_valid.cov(ddof=1)

# matriz de correlação, útil para interpretar a covariância
matriz_correlacao = Y_valid.corr(method="pearson")

display(media_respostas.to_frame("media"))
display(variancia_amostral.to_frame("variancia_amostral"))
display(matriz_covariancia)
display(matriz_correlacao)

In [ ]:
# ============================================================
# MÍNIMOS INDIVIDUAIS COM RESTRIÇÃO ESFÉRICA DO DOE
# x.T @ x <= R^2
# ============================================================



fit = resultados["quadratic_full"]
modelo = RSM_MODELO

# RAIO_DOE vem dos presets.

# Bounds auxiliares. A restrição principal é a esfera.
bounds = [(-RAIO_DOE, RAIO_DOE) for _ in FACTOR_COLS]

In [ ]:
def predizer_respostas_rsm(x, fit, factor_cols, response_cols, modelo="quadratic_full"):
    x = np.asarray(x, dtype=float).reshape(1, -1)

    df_x = pd.DataFrame(x, columns=factor_cols)

    Xnew = build_design_matrix(df_x, factor_cols, modelo=modelo)

    B = fit["B_coeficientes"].loc[Xnew.columns, response_cols].values

    y_pred = Xnew.values @ B

    return pd.Series(y_pred.ravel(), index=response_cols)


def resposta_convertida_minimizacao(y_fisico, response_cols, direcao):
    y_min = y_fisico.copy()

    for col in response_cols:
        if direcao[col] == "max":
            y_min[col] = -y_min[col]
        elif direcao[col] == "min":
            y_min[col] = y_min[col]
        else:
            raise ValueError(f"Direção inválida para {col}: {direcao[col]}")

    return y_min


def objetivo_individual(x, resposta, fit, factor_cols, response_cols, direcao, modelo):
    y_fisico = predizer_respostas_rsm(
        x=x,
        fit=fit,
        factor_cols=factor_cols,
        response_cols=response_cols,
        modelo=modelo
    )

    y_min = resposta_convertida_minimizacao(
        y_fisico,
        response_cols=response_cols,
        direcao=direcao
    )

    return float(y_min[resposta])

In [ ]:
def restricao_esfera_doe(x, raio=RAIO_DOE):
    x = np.asarray(x, dtype=float)
    return raio**2 - np.dot(x, x)


constraints = [
    {
        "type": "ineq",
        "fun": lambda x: restricao_esfera_doe(x, raio=RAIO_DOE)
    }
]

In [ ]:
def gerar_starts_esfera(factor_cols, raio=2.0, n_random=100, seed=42):
    rng = np.random.default_rng(seed)

    k = len(factor_cols)
    starts = []

    # Centro
    starts.append(np.zeros(k))

    # Pontos axiais: ±R em cada eixo
    for i in range(k):
        p1 = np.zeros(k)
        p2 = np.zeros(k)
        p1[i] = -raio
        p2[i] = raio
        starts.append(p1)
        starts.append(p2)

    # Pontos fatoriais -1/+1
    # Para k=4 e raio=2, esses pontos estão exatamente na esfera:
    # (-1)^2 + (-1)^2 + (-1)^2 + (-1)^2 = 4
    for p in product([-1.0, 1.0], repeat=k):
        p = np.array(p, dtype=float)
        if np.dot(p, p) <= raio**2 + 1e-12:
            starts.append(p)

    # Pontos aleatórios uniformes aproximados dentro da esfera
    for _ in range(n_random):
        v = rng.normal(size=k)
        v = v / np.linalg.norm(v)

        # raio aleatório corrigido para volume em k dimensões
        r = raio * (rng.random() ** (1.0 / k))

        starts.append(r * v)

    # Remove duplicados
    starts_unique = []
    for s in starts:
        if not any(np.allclose(s, u) for u in starts_unique):
            starts_unique.append(s)

    return starts_unique


starts = gerar_starts_esfera(
    FACTOR_COLS,
    raio=RAIO_DOE,
    n_random=N_STARTS_RANDOM_MINIMOS,
    seed=123
)

print(f"Número de starts: {len(starts)}")

# Conferência
normas = [np.dot(s, s) for s in starts]
print("Maior xTx nos starts:", max(normas))
print("Raio²:", RAIO_DOE**2)

In [ ]:
# ============================================================
# MÍNIMOS INDIVIDUAIS E EXPORTAÇÃO DA MATRIZ PAYOFF
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import minimize


# ------------------------------------------------------------
# 1) Executar as otimizações individuais
# ------------------------------------------------------------
_timer_payoff_otimizacao = iniciar_medicao_etapa()
_contador_payoff = {"avaliacoes_rsm": 0}

def _objetivo_payoff_contado(x, resposta):
    """Avalia o RSM e contabiliza uma chamada usada na construção da payoff."""
    _contador_payoff["avaliacoes_rsm"] += 1
    return objetivo_individual(
        x=x,
        resposta=resposta,
        fit=fit,
        factor_cols=FACTOR_COLS,
        response_cols=RESPONSE_COLS,
        direcao=direcao,
        modelo=modelo,
    )

resultados_minimos = []
resultados_exportacao = []

for resposta in RESPONSE_COLS:

    melhor = None
    melhor_sucesso = None

    for x0 in starts:

        res = minimize(
            fun=lambda x, resposta=resposta: _objetivo_payoff_contado(
                x=x,
                resposta=resposta,
            ),
            x0=x0,
            method="SLSQP",
            bounds=bounds,
            constraints=constraints,
            options={
                "ftol": 1e-12,
                "maxiter": 1000,
                "disp": False
            }
        )

        # Melhor resultado, independentemente do status do solver
        if melhor is None or res.fun < melhor.fun:
            melhor = res

        # Melhor resultado com convergência declarada
        if (
            res.success
            and (
                melhor_sucesso is None
                or res.fun < melhor_sucesso.fun
            )
        ):
            melhor_sucesso = res

    # Preferir o melhor resultado convergido
    if melhor_sucesso is not None:

        if (
            not melhor.success
            and melhor.fun < melhor_sucesso.fun - 1e-8
        ):
            print(
                f"Aviso [{resposta}]: o melhor valor absoluto veio de "
                f"um start sem sucesso "
                f"({melhor.fun:.6g} < {melhor_sucesso.fun:.6g}). "
                f"Usando o melhor resultado convergido."
            )

        melhor = melhor_sucesso

    # --------------------------------------------------------
    # Solução ótima no espaço dos fatores
    # --------------------------------------------------------
    x_star = np.asarray(melhor.x, dtype=float)
    xTx = float(x_star @ x_star)

    valores_x = dict(zip(FACTOR_COLS, x_star))

    # --------------------------------------------------------
    # Predições físicas de todas as respostas
    # --------------------------------------------------------
    # Reavaliação final no ótimo para preencher a linha completa da payoff.
    _contador_payoff["avaliacoes_rsm"] += 1
    y_fisico = predizer_respostas_rsm(
        x=x_star,
        fit=fit,
        factor_cols=FACTOR_COLS,
        response_cols=RESPONSE_COLS,
        modelo=modelo
    )

    y_min = resposta_convertida_minimizacao(
        y_fisico,
        response_cols=RESPONSE_COLS,
        direcao=direcao
    )

    # --------------------------------------------------------
    # Tabela completa para diagnóstico interno
    # --------------------------------------------------------
    linha_diagnostico = {
        "objetivo_otimizado": resposta,
        "sentido_original": direcao[resposta],
        "valor_fisico_original": float(y_fisico[resposta]),
        "valor_obj_min": float(y_min[resposta]),
        "xTx": xTx,
        "raio2": float(RAIO_DOE**2),
        "dentro_esfera": bool(
            xTx <= RAIO_DOE**2 + 1e-8
        ),
        "success": bool(melhor.success),
        "message": str(melhor.message),
        "fun": float(melhor.fun)
    }

    for fator, valor in valores_x.items():
        linha_diagnostico[f"{fator}_star"] = float(valor)

    for col in RESPONSE_COLS:
        linha_diagnostico[f"{col}_fisico"] = float(
            y_fisico[col]
        )

    for col in RESPONSE_COLS:
        linha_diagnostico[f"{col}_min"] = float(
            y_min[col]
        )

    resultados_minimos.append(linha_diagnostico)

    # --------------------------------------------------------
    # Tabela no formato solicitado para o Excel
    #
    # Apenas a resposta otimizada é preenchida.
    # As demais respostas permanecem em branco.
    # --------------------------------------------------------
    linha_exportacao = {
        "Resposta otimizada": resposta,
        "Cs": float(valores_x["cs"]),
        "f": float(valores_x["f"]),
        "md": float(valores_x["md"]),
        "xTx": xTx
    }

    # Inicializar todas as respostas como vazias
    for col in RESPONSE_COLS:
        linha_exportacao[col] = np.nan

    # Preencher somente a resposta otimizada
    linha_exportacao[resposta] = float(
        y_fisico[resposta]
    )

    resultados_exportacao.append(linha_exportacao)


# ------------------------------------------------------------
# 2) DataFrames
# ------------------------------------------------------------
df_minimos_individuais = pd.DataFrame(
    resultados_minimos
)

colunas_exportacao = [
    "Resposta otimizada",
    "Cs",
    "f",
    "md",
    "xTx",
    *RESPONSE_COLS
]

df_payoff_individual = pd.DataFrame(
    resultados_exportacao,
    columns=colunas_exportacao
)


# ------------------------------------------------------------
# 3) Salvar em Excel
# ------------------------------------------------------------
if "OUTPUT_DIR" in globals():
    pasta_saida = Path(OUTPUT_DIR)
else:
    pasta_saida = Path("resultados_cnbi")

pasta_saida.mkdir(
    parents=True,
    exist_ok=True
)

arquivo_excel = (
    pasta_saida
    / "minimos_individuais_payoff.xlsx"
)

with pd.ExcelWriter(arquivo_excel) as writer:

    # Planilha no formato solicitado
    df_payoff_individual.to_excel(
        writer,
        sheet_name="Payoff individual",
        index=False,
        na_rep=""
    )

    # Planilha adicional com todos os diagnósticos
    df_minimos_individuais.to_excel(
        writer,
        sheet_name="Diagnostico completo",
        index=False,
        na_rep=""
    )


# ------------------------------------------------------------
# 4) Exibir resultado
# ------------------------------------------------------------
display(df_payoff_individual)

print("=" * 70)
print("Matriz de mínimos individuais salva")
print("=" * 70)
print(f"Arquivo: {arquivo_excel.resolve()}")
print(f"Número de respostas: {len(df_payoff_individual)}")

_reg_payoff_otimizacao = registrar_custo_etapa(
    chave="01a_otimizacao_minimos_payoff",
    ordem=10,
    categoria="construcao_payoff",
    timer=_timer_payoff_otimizacao,
    avaliacoes_rsm=_contador_payoff["avaliacoes_rsm"],
    n_subproblemas=len(RESPONSE_COLS) * len(starts),
    n_combinacoes=len(RESPONSE_COLS),
    n_otimizacoes_individuais=len(RESPONSE_COLS),
    n_starts_por_objetivo=len(starts),
    n_execucoes_slsqp=len(RESPONSE_COLS) * len(starts),
    n_minimos_com_sucesso=int(df_minimos_individuais["success"].sum()),
    observacao=(
        "Inclui todas as chamadas do objetivo feitas pelo SLSQP, as "
        "reavaliações finais nos ótimos e a exportação dos mínimos."
    ),
)
print(
    "Custo da otimização individual da payoff: "
    f"{_reg_payoff_otimizacao['avaliacoes_rsm']} avaliações do RSM | "
    f"{_reg_payoff_otimizacao['tempo_parede_s']:.3f} s de parede"
)


In [ ]:
_timer_01b_montagem_payoff_min = iniciar_medicao_etapa()

# Verificação crítica: a payoff só deve ser montada se todos os mínimos individuais convergiram.
if not df_minimos_individuais["success"].fillna(False).astype(bool).all():
    display(df_minimos_individuais.loc[~df_minimos_individuais["success"].fillna(False).astype(bool),
                                      ["objetivo_otimizado", "success", "message", "fun"]])
    raise RuntimeError("Há mínimos individuais sem sucesso. Corrija a payoff antes de seguir para o CNBI.")

# Payoff em sentido de minimização
payoff_min = df_minimos_individuais[
    ["objetivo_otimizado"] + [f"{c}_min" for c in RESPONSE_COLS]
].copy()

payoff_min.columns = ["objetivo_otimizado"] + RESPONSE_COLS

display(payoff_min)

registrar_custo_etapa(
    chave="01b_montagem_payoff_min",
    ordem=11,
    categoria="construcao_payoff",
    timer=_timer_01b_montagem_payoff_min,
    observacao="Verificação de convergência e montagem da payoff em minimização.",
)


In [ ]:
_timer_01c_utopia_nadir_payoff = iniciar_medicao_etapa()

utopia = payoff_min[RESPONSE_COLS].min(axis=0)
nadir_aprox = payoff_min[RESPONSE_COLS].max(axis=0)

df_utopia_nadir = pd.DataFrame({
    "utopia_min": utopia,
    "nadir_aprox": nadir_aprox
})

display(df_utopia_nadir)

registrar_custo_etapa(
    chave="01c_utopia_nadir_payoff",
    ordem=12,
    categoria="construcao_payoff",
    timer=_timer_01c_utopia_nadir_payoff,
    observacao="Extração da utopia e do nadir aproximado a partir da payoff.",
)


In [ ]:
_timer_01d_escalonamento_payoff = iniciar_medicao_etapa()

# ============================================================
# PAYOFF ESCALONADA POR UTOPIA E NADIR
# 0 = melhor
# 1 = pior
# ============================================================

# Matriz numérica da payoff em minimização
P = payoff_min[RESPONSE_COLS].copy()

# Utopia e nadir aproximado a partir da própria payoff
utopia = P.min(axis=0)
nadir = P.max(axis=0)

# Escalonamento: quanto menor, melhor
payoff_scaled_values = (P - utopia) / (nadir - utopia)

# Evita divisão por zero caso algum objetivo tenha nadir = utopia
payoff_scaled_values = payoff_scaled_values.replace([np.inf, -np.inf], np.nan)
payoff_scaled_values = payoff_scaled_values.fillna(0.0)

# Monta tabela final
payoff_scaled = pd.concat(
    [payoff_min[["objetivo_otimizado"]], payoff_scaled_values],
    axis=1
)

display(payoff_scaled)

registrar_custo_etapa(
    chave="01d_escalonamento_payoff",
    ordem=13,
    categoria="construcao_payoff",
    timer=_timer_01d_escalonamento_payoff,
    observacao="Escalonamento da payoff para a faixa utopia–nadir.",
)


In [ ]:
# ============================================================
# HEATMAP DA MATRIZ PAYOFF ESCALONADA
# 0 = melhor / 1 = pior
# ============================================================

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# payoff_scaled deve ter:
# coluna "objetivo_otimizado" + colunas RESPONSE_COLS

payoff_heat = payoff_scaled.copy()

# Usa a coluna dos mínimos individuais como índice
payoff_heat = payoff_heat.set_index("objetivo_otimizado")

# Garante a ordem correta das colunas
payoff_heat = payoff_heat[RESPONSE_COLS]

# Rótulos das linhas
payoff_heat.index = [f"{idx}*" for idx in payoff_heat.index]

plt.figure(figsize=(14, 10))

sns.heatmap(
    payoff_heat,
    cmap="coolwarm",
    vmin=0,
    vmax=1,
    annot=True,
    fmt=".2f",
    annot_kws={"size": 12, "weight": "bold"},
    linewidths=0.5,
    linecolor="white",
    cbar_kws={
        "ticks": [0, 0.5, 1],
        "shrink": 0.85,
    },
    square=True
)

plt.ylabel("Individual optimization", fontsize=13, weight="bold")

plt.xticks(rotation=45, ha="right", fontsize=12)
plt.yticks(rotation=0, fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# SVD DO CHIM COMPLETO (ESPECTRO SINGULAR GLOBAL)
# Convenção: linhas = âncoras; colunas = respostas/objetivos.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_timer_svd_global = iniciar_medicao_etapa()

# ------------------------------------------------------------
# SVD do CHIM completo da payoff escalonada
# ------------------------------------------------------------
# CRÍTICO: cada linha da payoff é uma âncora, isto é, um vetor de respostas.
# Logo, as arestas do CHIM são diferenças de LINHAS, não de colunas.

P = payoff_scaled.copy()
if "objetivo_otimizado" in P.columns:
    P = P.set_index("objetivo_otimizado")

A_chim_global = P.loc[RESPONSE_COLS, RESPONSE_COLS].to_numpy(dtype=float)
E_chim_global = A_chim_global[1:, :] - A_chim_global[0:1, :]

_t_kernel_svd_global = perf_counter()
U_chim, S_chim_global, Vt_chim = np.linalg.svd(E_chim_global, full_matrices=False)
_tempo_kernel_svd_global_s = perf_counter() - _t_kernel_svd_global

energia_chim = np.cumsum(S_chim_global**2) / np.sum(S_chim_global**2)
df_svd_chim = pd.DataFrame({
    "componente": np.arange(1, len(S_chim_global) + 1),
    "valor_singular": S_chim_global,
    "valor_relativo": S_chim_global / S_chim_global[0],
    "energia_acumulada": energia_chim,
})

display(df_svd_chim)
print("Posto numérico do CHIM global:", np.linalg.matrix_rank(E_chim_global))

# ------------------------------------------------------------
# Gráficos
# ------------------------------------------------------------
plt.figure(figsize=(7, 4))
plt.plot(df_svd_chim["componente"], df_svd_chim["valor_singular"], marker="o")
plt.xlabel("Componente")
plt.ylabel("Valor singular")
plt.title("SVD do CHIM completo — arestas por linhas")
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(df_svd_chim["componente"], df_svd_chim["energia_acumulada"], marker="o")
plt.axhline(0.95, linestyle="--", label="95%")
plt.axhline(0.99, linestyle="--", label="99%")
plt.axhline(0.999, linestyle="--", label="99,9%")
plt.xlabel("Componente")
plt.ylabel("Energia acumulada")
plt.title("Energia acumulada da SVD do CHIM")
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend()
plt.tight_layout()
plt.show()


registrar_custo_etapa(
    chave="02_svd_chim_global",
    ordem=20,
    categoria="decomposicoes_svd",
    timer=_timer_svd_global,
    n_svd=1,
    n_combinacoes=1,
    tempo_kernel_svd_s=float(_tempo_kernel_svd_global_s),
    linhas_matriz=int(E_chim_global.shape[0]),
    colunas_matriz=int(E_chim_global.shape[1]),
    observacao="SVD explícita do CHIM global; o tempo da etapa inclui diagnósticos e gráficos.",
)


## Dimensão efetiva do CHIM — análise paralela com ruído propagado do RSM

A dimensão efetiva $d$ do CHIM global é estimada por um único critério: **análise
paralela do CHIM com o ruído de predição do próprio RSM propagado**
($\text{Var}[\hat y_j(x^*)] = MSE_j \cdot h(x^*)$). Compara-se o espectro singular
real do CHIM global ao espectro obtido de réplicas Monte Carlo de CHIMs de ruído
puro; $d$ é o número de componentes cujo valor singular real excede o percentil 95
do ruído. A partir de $d$, define-se a janela singular (piso geométrico $\sigma_{d+1}$
e teto $\sigma_1/\sigma_d$) usada para filtrar as combinações de objetivos do CNBI.


In [ ]:
# ============================================================
# DIMENSÃO EFETIVA DO CHIM — ANÁLISE PARALELA COM RUÍDO PROPAGADO DO RSM
# 1) Ruído de predição propagado para a payoff escalonada
# 2) Análise paralela: espectro real vs. espectro de CHIMs de ruído puro
# 3) Decisão final de d e definição da janela singular (piso geométrico + teto)
# ============================================================

_timer_analise_paralela = iniciar_medicao_etapa()

rng_mc = np.random.default_rng(SEED_MC_RUIDO)

# ------------------------------------------------------------
# 1) Ruído de predição propagado para a payoff escalonada
#    Var[yhat_j(x*_i)] = MSE_j * h(x*_i),  h = z'(X'X)^-1 z
# ------------------------------------------------------------
X_doe_design = build_design_matrix(df, FACTOR_COLS, modelo=RSM_MODELO)
XTX_inv_doe = np.linalg.pinv(X_doe_design.values.T @ X_doe_design.values)

met_rsm = resultados[RSM_MODELO]["metricas"].set_index("Resposta")
MSE_resp = met_rsm.loc[RESPONSE_COLS, "MSE"].to_numpy(dtype=float)

X_anc = df_minimos_individuais[[f"{c}_star" for c in FACTOR_COLS]].to_numpy(dtype=float)
Z_anc = build_design_matrix(
    pd.DataFrame(X_anc, columns=FACTOR_COLS), FACTOR_COLS, modelo=RSM_MODELO
).to_numpy(dtype=float)
h_anc = np.einsum("ij,jk,ik->i", Z_anc, XTX_inv_doe, Z_anc)

AMP_PAYOFF = (nadir[RESPONSE_COLS] - utopia[RESPONSE_COLS]).to_numpy(dtype=float)
AMP_PAYOFF = np.where(np.abs(AMP_PAYOFF) < 1e-12, 1.0, AMP_PAYOFF)

# Desvio-padrão de cada entrada A[i, j] da payoff ESCALONADA.
SD_A = np.sqrt(np.outer(h_anc, MSE_resp)) / AMP_PAYOFF[None, :]

P_tmp = payoff_scaled.copy()
if "objetivo_otimizado" in P_tmp.columns:
    P_tmp = P_tmp.set_index("objetivo_otimizado")
A_real = P_tmp.loc[RESPONSE_COLS, RESPONSE_COLS].to_numpy(dtype=float)

_t_svd_real_paralela = perf_counter()
s_real = np.linalg.svd(A_real[1:, :] - A_real[0:1, :], compute_uv=False)
_tempo_svd_real_paralela_s = perf_counter() - _t_svd_real_paralela

# ------------------------------------------------------------
# 2) Monte Carlo: espectro de ruído puro e análise paralela
# ------------------------------------------------------------
s_noise = np.empty((N_MC_RUIDO, len(s_real)))
_t_mc_indep = perf_counter()
for b in range(N_MC_RUIDO):
    ruido = rng_mc.normal(0.0, SD_A)
    s_noise[b] = np.linalg.svd(ruido[1:, :] - ruido[0:1, :], compute_uv=False)

_tempo_mc_indep_s = perf_counter() - _t_mc_indep

p95_noise = np.percentile(s_noise, 95, axis=0)
D_PARALELA_INDEP = int(np.sum(s_real > p95_noise))

# ------------------------------------------------------------
# 2b) SENSIBILIDADE: ruído CORRELACIONADO entre respostas
#     Os erros de predição das m respostas no MESMO ponto x*_i não são
#     independentes: compartilham a covariância residual Sigma_res do RSM
#     (Cov[e_i] = h(x*_i) * Sigma_res, na escala física). Ruído
#     correlacionado concentra energia em poucas direções (sigma_1 maior,
#     sigma_min menor), podendo deslocar d. As variâncias MARGINAIS são
#     idênticas ao caso independente (diag(Sigma_res) = MSE_j), então a
#     única diferença é a estrutura de correlação.
#     Simplificação declarada: linhas (âncoras) independentes entre si —
#     ignora-se a correlação induzida pelo beta-hat compartilhado.
# ------------------------------------------------------------
E_res_mat = resultados[RSM_MODELO]["residuos"].to_numpy(dtype=float)
n_doe, p_doe = X_doe_design.shape
Sigma_res = (E_res_mat.T @ E_res_mat) / (n_doe - p_doe)
Sigma_scaled = Sigma_res / np.outer(AMP_PAYOFF, AMP_PAYOFF)
L_sigma = np.linalg.cholesky(
    Sigma_scaled + 1e-12 * np.eye(len(RESPONSE_COLS))
)

rng_mc_corr = np.random.default_rng(SEED_MC_RUIDO + 1)
s_noise_corr = np.empty_like(s_noise)
_t_mc_corr = perf_counter()
for b in range(N_MC_RUIDO):
    Znoise = rng_mc_corr.normal(size=SD_A.shape)
    ruido_c = np.sqrt(h_anc)[:, None] * (Znoise @ L_sigma.T)
    s_noise_corr[b] = np.linalg.svd(
        ruido_c[1:, :] - ruido_c[0:1, :], compute_uv=False
    )

_tempo_mc_corr_s = perf_counter() - _t_mc_corr

p95_noise_corr = np.percentile(s_noise_corr, 95, axis=0)
D_PARALELA_CORR = int(np.sum(s_real > p95_noise_corr))

if str(RUIDO_MC_MODO).lower().startswith("corr"):
    D_PARALELA = D_PARALELA_CORR
else:
    D_PARALELA = D_PARALELA_INDEP

df_paralela_chim = pd.DataFrame({
    "componente": np.arange(1, len(s_real) + 1),
    "sigma_real": s_real,
    "ruido_p95_indep": p95_noise,
    "ruido_p95_corr": p95_noise_corr,
    "sinal_indep": s_real > p95_noise,
    "sinal_corr": s_real > p95_noise_corr,
})

print("Análise paralela do CHIM (ruído de predição do RSM propagado):")
display(df_paralela_chim.style.format({
    "sigma_real": "{:.4f}",
    "ruido_p95_indep": "{:.4f}",
    "ruido_p95_corr": "{:.4f}",
}))
print(f"d (ruído independente)   = {D_PARALELA_INDEP}")
print(f"d (ruído correlacionado) = {D_PARALELA_CORR}")
print(f"Modo adotado: {RUIDO_MC_MODO} -> d = {D_PARALELA}")
if D_PARALELA_INDEP != D_PARALELA_CORR:
    print("AVISO: d difere entre os modos de ruído. Reporte AMBOS na "
          "dissertação e justifique a escolha (análise de sensibilidade "
          "de H7); a janela singular abaixo usa o modo do preset.")

# ------------------------------------------------------------
# 3) Decisão final de d e definição da janela singular
#    Piso: resíduo geométrico bruto do CHIM global, sigma_(d+1).
#    Teto: número de condição de referência, sigma_1 / sigma_d.
# ------------------------------------------------------------
if D_CHIM_MANUAL is not None:
    D_FINAL = int(D_CHIM_MANUAL)
    ORIGEM_D = "manual (preset D_CHIM_MANUAL)"
else:
    D_FINAL = max(1, D_PARALELA)
    ORIGEM_D = "análise paralela do CHIM com ruído do RSM"

D_FINAL = min(D_FINAL, len(S_chim_global))
D_CHIM_REFERENCIA = D_FINAL

SIGMA_D_GLOBAL = float(S_chim_global[D_FINAL - 1])

# Piso: resíduo geométrico bruto do CHIM global (sigma_(d+1)).
SIGMA_PISO_RESIDUAL = (
    float(S_chim_global[D_FINAL])
    if D_FINAL < len(S_chim_global)
    else 0.0
)

Q_CHIM_CORTE_SVD = float(S_chim_global[0] / SIGMA_D_GLOBAL)
if USAR_JANELA_SINGULAR:
    Q_CHIM_CORTE = Q_CHIM_CORTE_SVD

df_resumo_janela_singular = pd.DataFrame([{
    "d_chim_referencia": D_CHIM_REFERENCIA,
    "origem_d": ORIGEM_D,
    "d_paralela_chim_ruido": D_PARALELA,
    "d_paralela_indep": D_PARALELA_INDEP,
    "d_paralela_corr": D_PARALELA_CORR,
    "modo_ruido_mc": RUIDO_MC_MODO,
    "sigma_1_global": float(S_chim_global[0]),
    "sigma_d_global": SIGMA_D_GLOBAL,
    "sigma_piso_residual": SIGMA_PISO_RESIDUAL,
    "teto_q_chim_svd": Q_CHIM_CORTE_SVD,
    "usar_janela_singular": USAR_JANELA_SINGULAR,
}])

print("=" * 70)
print("Janela singular do CHIM")
print("=" * 70)
print(f"d final = {D_FINAL}  |  critério: {ORIGEM_D}")
print(f"sigma_d = {SIGMA_D_GLOBAL:.6g}")
print(f"piso (sigma_(d+1)) = {SIGMA_PISO_RESIDUAL:.6g}")
print(f"teto q = sigma1/sigma_d = {Q_CHIM_CORTE_SVD:.6g}")

display(df_resumo_janela_singular.style.format({
    "sigma_1_global": "{:.6g}",
    "sigma_d_global": "{:.6g}",
    "sigma_piso_residual": "{:.6g}",
    "teto_q_chim_svd": "{:.6g}",
}))


registrar_custo_etapa(
    chave="03_analise_paralela_monte_carlo",
    ordem=30,
    categoria="analise_paralela_monte_carlo",
    timer=_timer_analise_paralela,
    n_svd=1 + 2 * int(N_MC_RUIDO),
    n_mc=2 * int(N_MC_RUIDO),
    n_combinacoes=2,
    n_mc_por_modelo=int(N_MC_RUIDO),
    n_modelos_ruido=2,
    tempo_svd_real_s=float(_tempo_svd_real_paralela_s),
    tempo_mc_indep_s=float(_tempo_mc_indep_s),
    tempo_mc_corr_s=float(_tempo_mc_corr_s),
    n_pseudoinversas=1,
    n_cholesky=1,
    observacao=(
        "Inclui os modelos de ruído independente e correlacionado. "
        "Cada réplica Monte Carlo executa uma SVD do CHIM de ruído."
    ),
)


In [ ]:
# ============================================================
# COMBINAÇÕES CNBI
# Objetivos tomados de 2 até n_x + 1
# Para seu caso:
# m_y = 10 respostas
# n_x = 4 fatores
# k = 2, 3, 4, 5
# ============================================================


# Configurações vêm dos presets.
m_y_cnbi = len(RESPONSE_COLS)
n_x_cnbi = len(FACTOR_COLS)


# ------------------------------------------------------------
# Número de pesos no simplex
# ------------------------------------------------------------

def n_pesos_simplex(k, delta):
    """
    Número de pontos beta no simplex de dimensão k com passo delta.

    Exemplo:
        delta = 0.25
        k=2 -> 5
        k=3 -> 15
        k=4 -> 35
        k=5 -> 70
    """

    p = int(round(1.0 / delta))

    if not math.isclose(p * delta, 1.0):
        raise ValueError(
            "delta precisa dividir 1 exatamente. "
            "Use valores que dividam 1 exatamente, como 0.50, 0.25, 0.20, 0.10 ou 0.05."
        )

    return math.comb(p + k - 1, k - 1)


# ------------------------------------------------------------
# Gerador das combinações
# ------------------------------------------------------------

def gerar_combinacoes_cnbi(
    m_y,
    n_x,
    nomes_objetivos,
    k_min=2,
    k_max=None,
    delta_preview=0.25
):
    """
    Gera todas as combinações de objetivos tomados de k,
    com k variando de k_min até n_x + 1.

    Para seu caso:
        m_y = 10
        n_x = 4
        k = 2, 3, 4, 5

    Total:
        C(10,2) + C(10,3) + C(10,4) + C(10,5)
        = 45 + 120 + 210 + 252
        = 627 combinações
    """

    if k_max is None:
        k_max = n_x + 1

    k_max = min(k_max, m_y)

    linhas = []
    combo_id = 1

    for k in range(k_min, k_max + 1):

        n_betas_preview = n_pesos_simplex(k, delta_preview)

        for combo in itertools.combinations(range(m_y), k):

            linha = {
                "combo_id": combo_id,
                "k": k,
                "n_betas_preview": n_betas_preview,
                "objetivos_idx_0based": combo,
                "objetivos_idx_1based": tuple(j + 1 for j in combo),
                "objetivos": tuple(nomes_objetivos[j] for j in combo),
            }

            # Matriz binária de seleção dos objetivos
            for j, nome in enumerate(nomes_objetivos):
                linha[nome] = 1 if j in combo else 0

            linhas.append(linha)
            combo_id += 1

    return pd.DataFrame(linhas)


# ------------------------------------------------------------
# Gerar combinações
# ------------------------------------------------------------

df_combinacoes = gerar_combinacoes_cnbi(
    m_y=m_y_cnbi,
    n_x=n_x_cnbi,
    nomes_objetivos=RESPONSE_COLS,
    k_min=2,
    k_max=n_x_cnbi + 1,
    delta_preview=DELTA_PREVIEW,
)

colunas_matriz = ["combo_id", "k", "n_betas_preview"] + RESPONSE_COLS
df_matriz_combinacoes = df_combinacoes[colunas_matriz].copy()


# ------------------------------------------------------------
# Resumo
# ------------------------------------------------------------

print("=" * 70)
print("Combinações CNBI")
print("=" * 70)
print(f"n_x = {n_x_cnbi}")
print(f"m_y = {m_y_cnbi}")
print("k mínimo = 2")
print(f"k máximo = n_x + 1 = {n_x_cnbi + 1}")
print(f"DELTA_PREVIEW = {DELTA_PREVIEW}")
print(f"Total de combinações = {len(df_combinacoes)}")
print(f"Total estimado de subproblemas NBI = {df_combinacoes['n_betas_preview'].sum()}")

print("\nNúmero de combinações e betas por tamanho k:")
display(
    df_combinacoes
    .groupby("k")
    .agg(
        n_combinacoes=("combo_id", "count"),
        n_betas_por_combo=("n_betas_preview", "first"),
        total_nbi=("n_betas_preview", "sum"),
    )
)

print("\nHead da tabela completa de combinações:")
display(df_combinacoes.head(10))

print("\nHead da matriz binária de combinações:")
display(df_matriz_combinacoes.head(10))


# ------------------------------------------------------------
# Salvar arquivos
# ------------------------------------------------------------

arquivo_combinacoes = OUTPUT_DIR / f"combinacoes_cnbi_m{m_y_cnbi}_nx{n_x_cnbi}.csv"
arquivo_matriz = OUTPUT_DIR / f"matriz_binaria_combinacoes_cnbi_m{m_y_cnbi}_nx{n_x_cnbi}.csv"

df_combinacoes.to_csv(arquivo_combinacoes, index=False)
df_matriz_combinacoes.to_csv(arquivo_matriz, index=False)

print(f"\nArquivos salvos em:")
print(arquivo_combinacoes.resolve())
print(arquivo_matriz.resolve())

In [ ]:
# ============================================================
# CRIAR df_combinacoes_com_q + JANELA SINGULAR
# ============================================================

import numpy as np
import pandas as pd
import math
import ast

_timer_preselecao_espectros = iniciar_medicao_etapa()


def parse_objetivos_combo(objetivos_combo):
    if isinstance(objetivos_combo, str):
        return list(ast.literal_eval(objetivos_combo))
    return list(objetivos_combo)


def calcular_espectro_chim_combo(payoff_scaled, objetivos_combo, eps=1e-12):
    P = payoff_scaled.copy()
    if "objetivo_otimizado" in P.columns:
        P = P.set_index("objetivo_otimizado")

    objetivos_combo = parse_objetivos_combo(objetivos_combo)
    A = P.loc[objetivos_combo, objetivos_combo].values.astype(float)

    # Convenção consistente com o CHIM global: linhas são âncoras.
    E = A[1:, :] - A[0:1, :]
    s = np.linalg.svd(E, compute_uv=False)

    sigma_max = float(np.max(s)) if len(s) else 0.0
    sigma_min = float(np.min(s)) if len(s) else 0.0
    q_chim = np.inf if sigma_min <= eps else sigma_max / sigma_min

    k = len(objetivos_combo)
    volume_chim = float(np.prod(s) / math.factorial(k - 1)) if len(s) else 0.0

    return {
        "q_chim": q_chim,
        "sigma_max_chim": sigma_max,
        "sigma_min_chim": sigma_min,
        "volume_chim": volume_chim,
    }


def adicionar_filtros_chim_combinacoes(df_combinacoes, payoff_scaled):
    linhas = []

    for _, row in df_combinacoes.iterrows():
        esp = calcular_espectro_chim_combo(
            payoff_scaled=payoff_scaled,
            objetivos_combo=row["objetivos"]
        )

        linha = row.to_dict()
        linha.update(esp)

        q = linha["q_chim"]
        smin = linha["sigma_min_chim"]

        linha["q_finito"] = bool(np.isfinite(q))
        linha["mantida_teto_q"] = bool(np.isfinite(q) and q <= Q_CHIM_CORTE_SVD)
        linha["mantida_piso_sigma"] = bool(smin > SIGMA_PISO_RESIDUAL)
        linha["mantida_janela_singular"] = bool(
            linha["mantida_teto_q"] and linha["mantida_piso_sigma"]
        )

        linhas.append(linha)

    return pd.DataFrame(linhas)


df_combinacoes_com_q = adicionar_filtros_chim_combinacoes(
    df_combinacoes=df_combinacoes,
    payoff_scaled=payoff_scaled
)

print("df_combinacoes_com_q criado:", df_combinacoes_com_q.shape)
print(f"Piso sigma_min > {SIGMA_PISO_RESIDUAL:.6g}")
print(f"Teto q_chim <= {Q_CHIM_CORTE_SVD:.6g}")

display(df_combinacoes_com_q.head())

display(
    df_combinacoes_com_q
    .groupby("k")
    .agg(
        combinacoes=("k", "size"),
        so_piso=("mantida_piso_sigma", "sum"),
        so_teto=("mantida_teto_q", "sum"),
        ambos_janela=("mantida_janela_singular", "sum"),
        sigma_min_mediano=("sigma_min_chim", "median"),
        q_mediano=("q_chim", "median"),
    )
    .reset_index()
)


registrar_custo_etapa(
    chave="04a_preselecao_espectros_subchims",
    ordem=40,
    categoria="pre_selecao_geometrica",
    timer=_timer_preselecao_espectros,
    n_svd=len(df_combinacoes),
    n_combinacoes=len(df_combinacoes),
    n_combinacoes_mantidas_janela=int(
        df_combinacoes_com_q["mantida_janela_singular"].sum()
    ),
    observacao=(
        "Uma SVD explícita por combinação candidata para obter sigma_max, "
        "sigma_min, q_chim e volume do sub-CHIM."
    ),
)


In [ ]:
# ============================================================
# FIGURE 4.4 — Singular-window diagnostics for all k
# Layout: 3 rows x 2 columns
#
# Rows:
#   k = 2, 3, 4
#
# Columns:
#   left  = q_C, log scale
#   right = sigma_min(E_C)
#
# Colors:
#   blue = accepted by the criterion shown in that panel
#   red  = rejected by the criterion shown in that panel
#
# Single legend:
#   accepted, rejected, q_max, sigma_floor
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from pathlib import Path

# ------------------------------------------------------------
# 1) Base data
# ------------------------------------------------------------
df_plot = df_combinacoes_com_q.copy()

COL_K = "k"
COL_Q = "q_chim"
COL_SIGMA = "sigma_min_chim"

required_cols = [COL_K, COL_Q, COL_SIGMA]
missing_cols = [c for c in required_cols if c not in df_plot.columns]

if missing_cols:
    raise ValueError(f"Missing columns in df_combinacoes_com_q: {missing_cols}")

df_plot[COL_K] = pd.to_numeric(df_plot[COL_K], errors="coerce")
df_plot[COL_Q] = pd.to_numeric(df_plot[COL_Q], errors="coerce")
df_plot[COL_SIGMA] = pd.to_numeric(df_plot[COL_SIGMA], errors="coerce")

df_plot = (
    df_plot
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=[COL_K, COL_Q, COL_SIGMA])
    .copy()
)

# q_C must be positive for log scale
df_plot = df_plot[df_plot[COL_Q] > 0].copy()

# Global cutoffs from notebook
q_cutoff = float(Q_CHIM_CORTE_SVD)
sigma_cutoff = float(SIGMA_PISO_RESIDUAL)

# ------------------------------------------------------------
# 2) Local acceptance criteria
# ------------------------------------------------------------
df_plot["accepted_q"] = df_plot[COL_Q] <= q_cutoff
df_plot["accepted_sigma"] = df_plot[COL_SIGMA] > sigma_cutoff
df_plot["accepted_window"] = df_plot["accepted_q"] & df_plot["accepted_sigma"]

print(f"q_max        = {q_cutoff:.6f}")
print(f"sigma_floor  = {sigma_cutoff:.6f}")
print(f"Total rows   = {len(df_plot)}")

# ------------------------------------------------------------
# 3) Plot settings
# ------------------------------------------------------------
k_values = [2, 3, 4]

blue = "tab:blue"
red = "tab:red"
line_color = "0.35"
cutoff_color = "black"

fig, axes = plt.subplots(
    nrows=3,
    ncols=2,
    figsize=(13.5, 12.0)
)

# ------------------------------------------------------------
# 4) Build panels
# ------------------------------------------------------------
for row, k_value in enumerate(k_values):
    d = df_plot[df_plot[COL_K] == k_value].copy()

    if d.empty:
        axes[row, 0].set_visible(False)
        axes[row, 1].set_visible(False)
        continue

    # --------------------------------------------------------
    # q_C panel: ordered by q_C
    # --------------------------------------------------------
    d_q = (
        d.sort_values(COL_Q, ascending=True, kind="mergesort")
         .reset_index(drop=True)
         .copy()
    )
    d_q["order_q"] = np.arange(1, len(d_q) + 1)

    d_q_acc = d_q[d_q["accepted_q"]]
    d_q_rej = d_q[~d_q["accepted_q"]]

    ax_q = axes[row, 0]

    ax_q.plot(
        d_q["order_q"],
        d_q[COL_Q],
        color=line_color,
        linewidth=1.3,
        alpha=0.85
    )

    ax_q.scatter(
        d_q_acc["order_q"],
        d_q_acc[COL_Q],
        s=34,
        color=blue,
        edgecolor="black",
        linewidth=0.25,
        zorder=4
    )

    ax_q.scatter(
        d_q_rej["order_q"],
        d_q_rej[COL_Q],
        s=34,
        color=red,
        edgecolor="black",
        linewidth=0.25,
        zorder=4
    )

    ax_q.axhline(
        q_cutoff,
        linestyle="--",
        linewidth=1.8,
        color=cutoff_color
    )

    ax_q.set_yscale("log")
    ax_q.set_ylabel(r"$q_C$ (log scale)")
    ax_q.set_xlabel(r"Combinations ordered by $q_C$")
    ax_q.grid(True, alpha=0.25, which="both")

    n = len(d)
    n_q_acc = int(d["accepted_q"].sum())
    n_window_acc = int(d["accepted_window"].sum())

    ax_q.set_title(
        rf"$k={k_value}$ | $q_C$ criterion "
        rf"({n_q_acc}/{n} accepted)"
    )

    # --------------------------------------------------------
    # sigma_min(E_C) panel: ordered by sigma_min(E_C)
    # --------------------------------------------------------
    d_sigma = (
        d.sort_values(COL_SIGMA, ascending=True, kind="mergesort")
         .reset_index(drop=True)
         .copy()
    )
    d_sigma["order_sigma"] = np.arange(1, len(d_sigma) + 1)

    d_sigma_acc = d_sigma[d_sigma["accepted_sigma"]]
    d_sigma_rej = d_sigma[~d_sigma["accepted_sigma"]]

    ax_sigma = axes[row, 1]

    ax_sigma.plot(
        d_sigma["order_sigma"],
        d_sigma[COL_SIGMA],
        color=line_color,
        linewidth=1.3,
        alpha=0.85
    )

    ax_sigma.scatter(
        d_sigma_acc["order_sigma"],
        d_sigma_acc[COL_SIGMA],
        s=34,
        color=blue,
        edgecolor="black",
        linewidth=0.25,
        zorder=4
    )

    ax_sigma.scatter(
        d_sigma_rej["order_sigma"],
        d_sigma_rej[COL_SIGMA],
        s=34,
        color=red,
        edgecolor="black",
        linewidth=0.25,
        zorder=4
    )

    ax_sigma.axhline(
        sigma_cutoff,
        linestyle="-.",
        linewidth=1.8,
        color=cutoff_color
    )

    ax_sigma.set_ylabel(r"$\sigma_{\min}(E_C)$")
    ax_sigma.set_xlabel(r"Combinations ordered by $\sigma_{\min}(E_C)$")
    ax_sigma.grid(True, alpha=0.25)

    n_sigma_acc = int(d["accepted_sigma"].sum())

    ax_sigma.set_title(
        rf"$k={k_value}$ | $\sigma_{{\min}}$ criterion "
        rf"({n_sigma_acc}/{n} accepted; final={n_window_acc})"
    )


# ------------------------------------------------------------
# 6) Single global legend
# ------------------------------------------------------------
legend_handles = [
    Line2D(
        [0], [0],
        marker="o",
        color="none",
        markerfacecolor=blue,
        markeredgecolor="black",
        markersize=8,
        label="Accepted"
    ),
    Line2D(
        [0], [0],
        marker="o",
        color="none",
        markerfacecolor=red,
        markeredgecolor="black",
        markersize=8,
        label="Rejected"
    ),
    Line2D(
        [0], [0],
        color=cutoff_color,
        linestyle="--",
        linewidth=1.8,
        label=rf"$q_{{\max}} = {q_cutoff:.4f}$"
    ),
    Line2D(
        [0], [0],
        color=cutoff_color,
        linestyle="-.",
        linewidth=1.8,
        label=rf"$\sigma_{{\mathrm{{floor}}}} = {sigma_cutoff:.4f}$"
    ),
]

fig.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.005),
    ncol=4,
    frameon=True,
    fontsize=10
)

plt.tight_layout(rect=[0, 0.04, 1, 0.97])

# ------------------------------------------------------------
# 7) Save figure
# ------------------------------------------------------------
out_dir = Path("figures_singular_window")
out_dir.mkdir(parents=True, exist_ok=True)

fpath = out_dir / "singular_window_diagnostics_3x2_logq.png"
fig.savefig(fpath, dpi=300, bbox_inches="tight")

print(f"Figure saved to: {fpath.resolve()}")

plt.show()

In [ ]:
# ============================================================
# SENSIBILIDADE DOS FILTROS DO CHIM
# q isolado, piso isolado e janela singular
# ============================================================

cortes = sorted(set([
    5, 10, 12.5, 20, 50, 100, 200, 500,
    round(Q_CHIM_CORTE_SVD, 6)
]))

linhas = []
for corte in cortes:
    df_tmp = df_combinacoes_com_q.copy()
    df_tmp["mantida_q_corte"] = df_tmp["q_finito"] & (df_tmp["q_chim"] <= corte)

    resumo = (
        df_tmp.groupby("k")
        .agg(
            combinacoes=("k", "size"),
            mantidas_q=("mantida_q_corte", "sum"),
            mantidas_piso=("mantida_piso_sigma", "sum"),
            mantidas_janela=("mantida_janela_singular", "sum"),
        )
        .reset_index()
    )

    resumo["corte_q"] = corte
    resumo["perc_mantidas_q"] = 100 * resumo["mantidas_q"] / resumo["combinacoes"]
    resumo["perc_mantidas_piso"] = 100 * resumo["mantidas_piso"] / resumo["combinacoes"]
    resumo["perc_mantidas_janela"] = 100 * resumo["mantidas_janela"] / resumo["combinacoes"]
    linhas.append(resumo)

df_sensibilidade_q = pd.concat(linhas, ignore_index=True)
display(df_sensibilidade_q)


In [ ]:
# ============================================================
# 2) Plotar distribuição do q_chim por k
# ============================================================



def plotar_distribuicao_q_por_k(df, coluna_q="q_chim", corte=10.0):
    """
    Plota e resume a distribuição do q_chim do CHIM por k.

    df precisa conter:
      - k
      - q_chim

    Regra:
      manter q_chim <= corte
    """

    df = df.copy()

    if "k" not in df.columns:
        raise ValueError("A coluna 'k' não existe no dataframe.")

    if coluna_q not in df.columns:
        raise ValueError(f"A coluna '{coluna_q}' não existe no dataframe.")

    df["k"] = pd.to_numeric(df["k"], errors="coerce")
    df[coluna_q] = pd.to_numeric(df[coluna_q], errors="coerce")

    df = df.dropna(subset=["k", coluna_q]).copy()
    df["k"] = df["k"].astype(int)

    df["q_finito"] = np.isfinite(df[coluna_q])
    df_finito = df[df["q_finito"]].copy()

    if df_finito.empty:
        raise ValueError(f"A coluna '{coluna_q}' não possui valores finitos.")

    resumo = (
        df_finito.groupby("k")[coluna_q]
        .describe(percentiles=[0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95])
        .reset_index()
    )

    resumo_nao_finitos = (
        df.groupby("k")
        .agg(
            combinacoes=("k", "size"),
            q_finitos=("q_finito", "sum"),
            q_nao_finitos=("q_finito", lambda x: (~x).sum()),
        )
        .reset_index()
    )

    print("Resumo descritivo por k:")
    display(resumo)

    print("Resumo de valores finitos/não finitos:")
    display(resumo_nao_finitos)

    # Histograma normal
    for k_atual in sorted(df["k"].unique()):
        d = df_finito[df_finito["k"] == k_atual].copy()

        if d.empty:
            print(f"k={k_atual}: sem valores finitos.")
            continue

        plt.figure(figsize=(7, 4))
        plt.hist(d[coluna_q], bins=20)
        plt.axvline(corte, linestyle="--", label=f"corte q = {corte}")
        plt.xlabel(coluna_q)
        plt.ylabel("Frequência")
        plt.title(f"Distribuição do q_chim do CHIM | k={k_atual}")
        plt.legend()
        plt.tight_layout()
        plt.show()

    # Histograma log
    for k_atual in sorted(df["k"].unique()):
        d = df_finito[
            (df_finito["k"] == k_atual) &
            (df_finito[coluna_q] > 0)
        ].copy()

        if d.empty:
            continue

        plt.figure(figsize=(7, 4))
        plt.hist(d[coluna_q], bins=20)
        plt.axvline(corte, linestyle="--", label=f"corte q = {corte}")
        plt.xscale("log")
        plt.xlabel(f"{coluna_q} em escala log")
        plt.ylabel("Frequência")
        plt.title(f"Distribuição log do q_chim do CHIM | k={k_atual}")
        plt.legend()
        plt.tight_layout()
        plt.show()

    # Curva ordenada
    for k_atual in sorted(df["k"].unique()):
        d = df_finito[df_finito["k"] == k_atual].copy()
        d = d.sort_values(coluna_q, ascending=True).reset_index(drop=True)

        if d.empty:
            continue

        plt.figure(figsize=(7, 4))
        plt.plot(np.arange(1, len(d) + 1), d[coluna_q], marker="o")
        plt.axhline(corte, linestyle="--", label=f"corte q = {corte}")
        plt.xlabel("Combinações ordenadas")
        plt.ylabel(coluna_q)
        plt.title(f"Crescimento do q_chim do CHIM | k={k_atual}")
        plt.legend()
        plt.tight_layout()
        plt.show()

    # Curva ordenada log
    for k_atual in sorted(df["k"].unique()):
        d = df_finito[
            (df_finito["k"] == k_atual) &
            (df_finito[coluna_q] > 0)
        ].copy()

        d = d.sort_values(coluna_q, ascending=True).reset_index(drop=True)

        if d.empty:
            continue

        plt.figure(figsize=(7, 4))
        plt.plot(np.arange(1, len(d) + 1), d[coluna_q], marker="o")
        plt.axhline(corte, linestyle="--", label=f"corte q = {corte}")
        plt.yscale("log")
        plt.xlabel("Combinações ordenadas")
        plt.ylabel(f"{coluna_q} em escala log")
        plt.title(f"Crescimento log do q_chim do CHIM | k={k_atual}")
        plt.legend()
        plt.tight_layout()
        plt.show()

    # Impacto do corte
    df["mantida"] = df["q_finito"] & (df[coluna_q] <= corte)

    resumo_corte = (
        df.groupby("k")
        .agg(
            combinacoes=("k", "size"),
            mantidas=("mantida", "sum"),
            q_nao_finitos=("q_finito", lambda x: (~x).sum()),
        )
        .reset_index()
    )

    estat_q = (
        df_finito.groupby("k")[coluna_q]
        .agg(
            q_min="min",
            q_p10=lambda x: x.quantile(0.10),
            q_p25=lambda x: x.quantile(0.25),
            q_mediano="median",
            q_p75=lambda x: x.quantile(0.75),
            q_p90=lambda x: x.quantile(0.90),
            q_max="max",
        )
        .reset_index()
    )

    resumo_corte = resumo_corte.merge(estat_q, on="k", how="left")

    resumo_corte["descartadas"] = (
        resumo_corte["combinacoes"] - resumo_corte["mantidas"]
    )

    resumo_corte["perc_mantidas"] = (
        resumo_corte["mantidas"] / resumo_corte["combinacoes"]
    )

    resumo_corte["perc_descartadas"] = (
        resumo_corte["descartadas"] / resumo_corte["combinacoes"]
    )

    print("Impacto do corte:")
    display(resumo_corte)

    return resumo, resumo_corte


resumo_q, resumo_corte_q = plotar_distribuicao_q_por_k(
    df_combinacoes_com_q,
    coluna_q="q_chim",
    corte=Q_CHIM_CORTE
)

In [ ]:
# ============================================================
# GRÁFICO: PERCENTUAL DE COMBINAÇÕES MANTIDAS POR k
# Usa resumo_corte_q gerado pelo filtro q_chim
# ============================================================

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df_plot = resumo_corte_q.copy()

# Percentuais em %
df_plot["perc_mantidas_pct"] = 100 * df_plot["perc_mantidas"]
df_plot["perc_descartadas_pct"] = 100 * df_plot["perc_descartadas"]

plt.figure(figsize=(8, 5))

plt.plot(
    df_plot["k"],
    df_plot["perc_mantidas_pct"],
    marker="o",
    linewidth=2,
    label="Combinações mantidas"
)

for _, row in df_plot.iterrows():
    plt.text(
        row["k"],
        row["perc_mantidas_pct"] + 2,
        f"{row['perc_mantidas_pct']:.2f}%",
        ha="center",
        fontsize=11,
        weight="bold"
    )

plt.xlabel("Número de objetivos na combinação (k)", fontsize=12)
plt.ylabel("Percentual de combinações mantidas (%)", fontsize=12)
plt.title(
    f"Percentual de combinações mantidas por k\nFiltro: q_chim ≤ {Q_CHIM_CORTE}",
    fontsize=14,
    weight="bold"
)

plt.xticks(df_plot["k"])
plt.ylim(-5, 105)
plt.grid(True, linestyle="--", alpha=0.4)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
_timer_preselecao_filtro = iniciar_medicao_etapa()

# ============================================================
# FILTRAR COMBINAÇÕES CNBI
# Preferência: janela singular do CHIM.
# Fallback: corte por q_chim.
# ============================================================

if USAR_JANELA_SINGULAR and "mantida_janela_singular" in df_combinacoes_com_q.columns:
    mascara_filtro_chim = df_combinacoes_com_q["mantida_janela_singular"].fillna(False).astype(bool)
    criterio_filtro_chim = (
        f"janela singular: sigma_min > {SIGMA_PISO_RESIDUAL:.6g} "
        f"e q_chim <= {Q_CHIM_CORTE_SVD:.6g}"
    )
else:
    mascara_filtro_chim = (
        np.isfinite(df_combinacoes_com_q["q_chim"]) &
        (df_combinacoes_com_q["q_chim"] <= Q_CHIM_CORTE)
    )
    criterio_filtro_chim = f"q_chim <= {Q_CHIM_CORTE}"


df_combinacoes_filtradas = df_combinacoes_com_q[mascara_filtro_chim].copy()

df_combinacoes_filtradas = df_combinacoes_filtradas.sort_values(
    ["k", "q_chim", "sigma_min_chim"],
    ascending=[True, True, False]
).reset_index(drop=True)

print("Critério:", criterio_filtro_chim)
print("Combinações totais:", len(df_combinacoes_com_q))
print("Combinações filtradas:", len(df_combinacoes_filtradas))
print("Descartadas:", len(df_combinacoes_com_q) - len(df_combinacoes_filtradas))

display(
    df_combinacoes_filtradas
    .groupby("k")
    .agg(
        mantidas=("combo_id", "count"),
        q_min=("q_chim", "min"),
        q_mediano=("q_chim", "median"),
        q_max=("q_chim", "max"),
        sigma_min_min=("sigma_min_chim", "min"),
        sigma_min_mediano=("sigma_min_chim", "median"),
        volume_mediano=("volume_chim", "median"),
    )
)


registrar_custo_etapa(
    chave="04b_aplicacao_filtro_geometrico",
    ordem=41,
    categoria="pre_selecao_geometrica",
    timer=_timer_preselecao_filtro,
    n_combinacoes=len(df_combinacoes_com_q),
    n_combinacoes_mantidas=len(df_combinacoes_filtradas),
    n_combinacoes_descartadas=len(df_combinacoes_com_q) - len(df_combinacoes_filtradas),
    observacao="Aplicação da janela singular e ordenação das combinações selecionadas.",
)


In [ ]:
# ============================================================
# GERAR BETAS POR k E ESTIMAR CUSTO DO CNBI
# Usa DELTA_POR_K definido nos presets.
# ============================================================


def gerar_betas_simplex(k, delta=0.25):
    """
    Gera pesos beta no simplex:
        beta_i >= 0
        sum(beta_i) = 1
    """
    p = int(round(1 / delta))
    if not np.isclose(p * delta, 1.0):
        raise ValueError("DELTA precisa dividir 1 exatamente.")

    betas = []
    for comb in itertools.product(range(p + 1), repeat=k):
        if sum(comb) == p:
            betas.append(np.array(comb, dtype=float) / p)
    return np.array(betas)


betas_por_k = {}
for k in sorted(df_combinacoes_filtradas["k"].unique()):
    k_int = int(k)
    if k_int not in DELTA_POR_K:
        raise ValueError(f"Delta não definido para k={k_int} em DELTA_POR_K.")
    betas_por_k[k_int] = gerar_betas_simplex(k=k_int, delta=DELTA_POR_K[k_int])

print("Betas por k:")
for k, B in betas_por_k.items():
    print(f"k={k} | delta={DELTA_POR_K[k]:.2f} | n_betas={len(B)}")

custo_filtrado = 0
for k, d in df_combinacoes_filtradas.groupby("k"):
    k_int = int(k)
    n_combo = len(d)
    n_beta = len(betas_por_k[k_int])
    custo_filtrado += n_combo * n_beta
    print(f"k={k_int}: {n_combo} combinações × {n_beta} betas = {n_combo*n_beta}")

print("Total de subproblemas CNBI:", custo_filtrado)


In [ ]:
# ============================================================
# CNBI RSM OTIMIZADO — TODAS AS COMBINAÇÕES MANTIDAS
# Com:
# - todas as combinações filtradas por q_chim
# - jacobiano analítico do RSM
# - paralelismo por combinação
# - checkpoint progressivo
# - warm start entre betas da mesma combinação
# ============================================================



_timer_cnbi = iniciar_medicao_etapa()

# ============================================================
# 1) CONFIGURAÇÕES
# ============================================================

fit = resultados[RSM_MODELO]
modelo = RSM_MODELO

# usar TODAS as combinações mantidas pelo filtro q_chim
df_combinacoes_execucao = df_combinacoes_filtradas.copy()

# Identificador do preset de delta adaptativo para checkpoint separado
delta_tag = "adapt_" + "_".join(
    f"k{k}_{str(v).replace('.', 'p')}"
    for k, v in sorted(DELTA_POR_K.items())
)
filtro_tag = f"janela_singular_d{D_CHIM_REFERENCIA}" if USAR_JANELA_SINGULAR else f"q{Q_CHIM_CORTE:.6g}"

# Versão do solver. O sufixo força um checkpoint novo e impede que os
# quatro resultados inválidos da versão anterior sejam reaproveitados.
SOLVER_TAG_NBI = "anchor_safe_v3_budget_wall"
TOL_ESFERA_NBI = 1e-8
TOL_BETA_VERTICE = 1e-12
path_checkpoint = CHECKPOINT_DIR / (
    f"checkpoint_cnbi_rsm_.csv"
)

bounds_z = [(-RAIO_DOE, RAIO_DOE) for _ in FACTOR_COLS] + [(-10.0, 10.0)]

print("=" * 80)
print("CNBI RSM otimizado")
print("=" * 80)
print(f"Combinações mantidas usadas: {len(df_combinacoes_execucao)}")
print(f"DELTA_POR_K = {DELTA_POR_K}")
print(f"N_JOBS = {N_JOBS}")
print(f"Checkpoint = {path_checkpoint}")

display(
    df_combinacoes_execucao
    .groupby("k")
    .agg(
        mantidas=("combo_id", "count"),
        q_min=("q_chim", "min"),
        q_mediano=("q_chim", "median"),
        q_max=("q_chim", "max"),
        volume_mediano=("volume_chim", "median")
    )
)


# ============================================================
# 2) COEFICIENTES RSM E DERIVADAS ANALÍTICAS
# ============================================================

B_coef = fit["B_coeficientes"].loc[:, RESPONSE_COLS].copy()
design_cols = B_coef.index.tolist()
B_np = B_coef.values.astype(float)
RESPONSE_IDX = {c: i for i, c in enumerate(RESPONSE_COLS)}
SIGN = np.array([-1.0 if str(direcao[c]).lower().startswith("max") else 1.0 for c in RESPONSE_COLS], dtype=float)
UTO = utopia[RESPONSE_COLS].to_numpy(dtype=float) if hasattr(utopia, "to_numpy") else np.asarray([utopia[c] for c in RESPONSE_COLS], dtype=float)
NAD = nadir[RESPONSE_COLS].to_numpy(dtype=float) if hasattr(nadir, "to_numpy") else np.asarray([nadir[c] for c in RESPONSE_COLS], dtype=float)
AMP = NAD - UTO
AMP = np.where(np.abs(AMP) < 1e-12, 1.0, AMP)


def design_vector_quadratic(x):
    x = np.asarray(x, dtype=float)
    vals = {"Intercepto": 1.0}

    for i, c in enumerate(FACTOR_COLS):
        vals[c] = x[i]

    for i, c in enumerate(FACTOR_COLS):
        vals[f"{c}^2"] = x[i] ** 2

    for i, a in enumerate(FACTOR_COLS):
        for j, b in enumerate(FACTOR_COLS):
            if j > i:
                vals[f"{a}:{b}"] = x[i] * x[j]

    return np.array([vals.get(c, 0.0) for c in design_cols], dtype=float)


def jac_design_quadratic(x):
    x = np.asarray(x, dtype=float)
    n = len(FACTOR_COLS)

    J = np.zeros((len(design_cols), n), dtype=float)
    col_pos = {c: i for i, c in enumerate(design_cols)}

    # lineares
    for i, c in enumerate(FACTOR_COLS):
        if c in col_pos:
            J[col_pos[c], i] = 1.0

    # quadráticos
    for i, c in enumerate(FACTOR_COLS):
        termo = f"{c}^2"
        if termo in col_pos:
            J[col_pos[termo], i] = 2.0 * x[i]

    # interações
    for i, a in enumerate(FACTOR_COLS):
        for j, b in enumerate(FACTOR_COLS):
            if j > i:
                termo = f"{a}:{b}"
                if termo in col_pos:
                    J[col_pos[termo], i] = x[j]
                    J[col_pos[termo], j] = x[i]

    return J


def predizer_scaled_jac_np(x):
    """Predição RSM em numpy puro para uso no loop interno do SLSQP."""
    x = np.asarray(x, dtype=float)
    xv = design_vector_quadratic(x)
    Jx = jac_design_quadratic(x)

    y_fisico = xv @ B_np                    # shape = (m,)
    J_fisico = (Jx.T @ B_np).T              # shape = (m, n_fatores)

    y_min = SIGN * y_fisico
    J_min = SIGN[:, None] * J_fisico

    y_scaled = (y_min - UTO) / AMP
    J_scaled = J_min / AMP[:, None]

    return y_fisico, y_min, y_scaled, J_scaled


def predizer_ymin_scaled_jac(x):
    """Wrapper com pandas apenas para saída/relatórios fora do miolo crítico."""
    y_fisico_arr, y_min_arr, y_scaled_arr, J_scaled_arr = predizer_scaled_jac_np(x)
    y_fisico = pd.Series(y_fisico_arr, index=RESPONSE_COLS)
    y_min = pd.Series(y_min_arr, index=RESPONSE_COLS)
    y_scaled = pd.Series(y_scaled_arr, index=RESPONSE_COLS)
    J_scaled = pd.DataFrame(J_scaled_arr, index=RESPONSE_COLS, columns=FACTOR_COLS)
    return y_fisico, y_min, y_scaled, J_scaled


# ============================================================
# 2b) ORÇAMENTO COMPUTACIONAL — CONTADOR DE AVALIAÇÕES DO RSM
# ============================================================
# Estratégia: envolver a própria função de avaliação do RSM (não confiar
# apenas em nit/nfev/njev do scipy) para contar toda chamada ao modelo,
# de forma agnóstica ao solver. Isso permite comparar o orçamento gasto
# aqui com o de outros algoritmos (ex.: NSGA-III, MOEA/D) na mesma moeda:
# número de avaliações do modelo RSM, não iterações nem tempo de parede.
from functools import wraps


class ContadorAvaliacoes:
    """Conta quantas vezes uma função de avaliação foi chamada."""

    def __init__(self):
        self.n_avaliacoes = 0

    def reset(self):
        self.n_avaliacoes = 0

    def wrap(self, func):
        @wraps(func)
        def _wrapped(*args, **kwargs):
            self.n_avaliacoes += 1
            return func(*args, **kwargs)
        return _wrapped


contador_rsm = ContadorAvaliacoes()
predizer_scaled_jac_np = contador_rsm.wrap(predizer_scaled_jac_np)
# A partir daqui, toda chamada a predizer_scaled_jac_np (feita dentro de
# eq_nbi e jac_eq_nbi, únicos pontos do solver que de fato avaliam o RSM)
# incrementa contador_rsm.n_avaliacoes. objetivo/jac_objetivo e
# restricao_esfera/jac_esfera NÃO avaliam o RSM e não são contados.
#
# CONVENÇÃO DE ORÇAMENTO (documentar na Seção 3.10 da dissertação):
# 1 avaliação = 1 ponto x m respostas x (valor + Jacobiano analítico).
# Como fun (eq_nbi) e jac (jac_eq_nbi) do SLSQP chamam o preditor
# SEPARADAMENTE no mesmo ponto, cada iteração conta ~2 avaliações — isso
# INFLA o orçamento atribuído ao CNBI (conservador contra o método
# proposto). Em contrapartida, cada chamada carrega o Jacobiano completo,
# informação de primeira ordem que NSGA-III/MOEA-D não recebem pela mesma
# "moeda". Ambos os efeitos devem ser declarados no texto.
# FORA apenas do subtotal interno do solver CNBI: mínimos individuais da
# payoff, Monte Carlo e SVDs. Esses custos entram no orçamento consolidado
# criado após o pós-processamento.


# ============================================================
# 3) PAYOFF / CHIM CACHE
# ============================================================

P_scaled = payoff_scaled.copy()
if "objetivo_otimizado" in P_scaled.columns:
    P_scaled = P_scaled.set_index("objetivo_otimizado")

# Coordenadas, no espaço de decisão, dos mínimos individuais usados para
# construir cada linha da payoff. Elas são indispensáveis nos vértices do
# simplex: beta=e_i deve começar exatamente no ótimo individual i.
X_PAYOFF = (
    df_minimos_individuais
    .set_index("objetivo_otimizado")
    [[f"{c}_star" for c in FACTOR_COLS]]
    .astype(float)
)


def calcular_chim_local(objetivos_combo):
    A = P_scaled.loc[objetivos_combo, objetivos_combo].values.astype(float)

    E = A[1:, :] - A[0:1, :]
    _, _, Vt = np.linalg.svd(E, full_matrices=True)

    normal = Vt[-1, :]
    normal = normal / np.linalg.norm(normal)

    centro = A.mean(axis=0)
    proj_utopia = float(np.dot(normal, -centro))
    if abs(proj_utopia) < 1e-9:
        raise RuntimeError(
            "Orientação da normal indefinida (centroide do sub-CHIM ~ origem "
            "na escala utopia/nadir): combinação geometricamente degenerada "
            "que escapou à janela singular — investigar antes de prosseguir."
        )
    if proj_utopia < 0:
        normal = -normal

    return A, normal


combo_cache = {}

for _, row in df_combinacoes_execucao.iterrows():
    objetivos_combo = row["objetivos"]
    if isinstance(objetivos_combo, str):
        objetivos_combo = ast.literal_eval(objetivos_combo)

    objetivos_combo = tuple(objetivos_combo)
    A, normal = calcular_chim_local(list(objetivos_combo))
    X_anchor = X_PAYOFF.loc[list(objetivos_combo)].to_numpy(dtype=float)

    # A esfera é convexa; portanto, combinações convexas das âncoras também
    # permanecem na região experimental. Verificação explícita para impedir
    # que uma payoff numericamente inviável contamine o NBI.
    xTx_anchor = np.einsum("ij,ij->i", X_anchor, X_anchor)
    if np.any(xTx_anchor > RAIO_DOE**2 + TOL_ESFERA_NBI):
        ruins = np.where(xTx_anchor > RAIO_DOE**2 + TOL_ESFERA_NBI)[0]
        raise RuntimeError(
            f"Âncora(s) da payoff fora da esfera na combinação {objetivos_combo}: "
            f"índices {ruins.tolist()}, xTx={xTx_anchor[ruins].tolist()}"
        )

    combo_cache[int(row["combo_id"])] = {
        "objetivos_combo": objetivos_combo,
        "k": int(row["k"]),
        "A": A,
        "X_anchor": X_anchor,
        "normal": normal,
        "q_chim": float(row["q_chim"]),
        "volume_chim": float(row["volume_chim"]),
    }


# ============================================================
# 4) BETAS
# ============================================================

# (CORREÇÃO: removida a redefinição duplicada de gerar_betas_simplex que
#  existia aqui — a definição única fica na célula 'GERAR BETAS POR k E
#  ESTIMAR CUSTO DO CNBI', que valida DELTA_POR_K estritamente.)

def ordenar_betas_vizinho_mais_proximo(betas):
    betas = np.asarray(betas, dtype=float)
    n = len(betas)

    if n <= 2:
        return [(i + 1, betas[i]) for i in range(n)]

    restantes = set(range(n))
    ordem = [0]
    restantes.remove(0)
    atual = 0

    while restantes:
        candidatos = np.array(list(restantes))
        d = np.linalg.norm(betas[candidatos] - betas[atual], axis=1)
        prox = int(candidatos[np.argmin(d)])
        ordem.append(prox)
        restantes.remove(prox)
        atual = prox

    return [(i + 1, betas[i]) for i in ordem]


# (CORREÇÃO: havia aqui uma redefinição de betas_por_k com fallback
#  silencioso DELTA_POR_K.get(k, 0.25), que podia divergir da versão
#  estrita da célula de custo. Agora reutiliza-se a definição única.)
if "betas_por_k" not in globals():
    raise RuntimeError(
        "betas_por_k não definido — execute antes a célula "
        "'GERAR BETAS POR k E ESTIMAR CUSTO DO CNBI'."
    )
_ks_faltantes = [int(k) for k in df_combinacoes_execucao["k"].unique()
                 if int(k) not in betas_por_k]
if _ks_faltantes:
    raise ValueError(f"betas_por_k sem k={_ks_faltantes}; verifique DELTA_POR_K.")

print("\nBetas por k (definidos na célula de custo):")
for k, B in betas_por_k.items():
    print(f"k={k} | delta={DELTA_POR_K[k]:.2f} | n_betas={len(B)}")


# ============================================================
# 5) SOLVER DE UM BETA — VERSÃO ANCHOR-SAFE
# ============================================================

def _indice_vertice_simplex(beta, tol=TOL_BETA_VERTICE):
    """Retorna i quando beta é o vértice e_i; caso contrário, retorna None."""
    beta = np.asarray(beta, dtype=float)
    i = int(np.argmax(beta))
    if (
        abs(beta[i] - 1.0) <= tol
        and np.max(np.abs(np.delete(beta, i))) <= tol
        and abs(beta.sum() - 1.0) <= tol
    ):
        return i
    return None


def _projetar_t_na_reta_nbi(x, phi, normal, idx_combo):
    """Melhor t escalar, por projeção ortogonal, para um chute x."""
    _, _, y_scaled, _ = predizer_scaled_jac_np(x)
    y_sub = y_scaled[idx_combo].astype(float)
    return float(np.dot(y_sub - phi, normal))


def _chave_start(z):
    return tuple(np.round(np.asarray(z, dtype=float), 12))


def resolver_nbi_beta_cached(combo_id, beta_id, beta, z0_warm=None):
    cache = combo_cache[int(combo_id)]

    objetivos_combo = list(cache["objetivos_combo"])
    k = cache["k"]
    idx_combo = np.array([RESPONSE_IDX[c] for c in objetivos_combo], dtype=int)
    A = cache["A"]
    X_anchor = cache["X_anchor"]
    normal = cache["normal"]

    beta = np.asarray(beta, dtype=float)
    phi = beta @ A

    def objetivo(z):
        return -z[-1]

    def jac_objetivo(z):
        g = np.zeros(len(FACTOR_COLS) + 1)
        g[-1] = -1.0
        return g

    def restricao_esfera(z):
        x = z[:-1]
        return RAIO_DOE**2 - np.dot(x, x)

    def jac_esfera(z):
        x = z[:-1]
        g = np.zeros(len(FACTOR_COLS) + 1)
        g[:-1] = -2.0 * x
        return g

    def eq_nbi(z):
        x = z[:-1]
        t = z[-1]

        _, _, y_scaled, _ = predizer_scaled_jac_np(x)
        y_sub = y_scaled[idx_combo].astype(float)

        return y_sub - (phi + t * normal)

    def jac_eq_nbi(z):
        x = z[:-1]

        _, _, _, J_scaled = predizer_scaled_jac_np(x)
        J_sub = J_scaled[idx_combo, :].astype(float)

        J = np.zeros((k, len(FACTOR_COLS) + 1))
        J[:, :-1] = J_sub
        J[:, -1] = -normal

        return J

    constraints = [
        {"type": "ineq", "fun": restricao_esfera, "jac": jac_esfera},
        {"type": "eq", "fun": eq_nbi, "jac": jac_eq_nbi},
    ]

    contador_rsm.reset()

    # --------------------------------------------------------
    # CASO-CHAVE: vértice do simplex = ótimo individual
    # --------------------------------------------------------
    # Para beta=e_i, phi é exatamente a linha i da payoff. Assim,
    # (x_anchor_i, t=0) satisfaz a igualdade do NBI por construção.
    # Não se pede ao SLSQP que "redescubra" uma solução que pode estar com
    # a restrição esférica ativa; isso elimina as quatro falhas observadas
    # nas combinações triplas que continham ROI.
    idx_vertice = _indice_vertice_simplex(beta)
    endpoint_payoff = idx_vertice is not None

    if endpoint_payoff:
        x_star = np.asarray(X_anchor[idx_vertice], dtype=float).copy()
        xTx = float(np.dot(x_star, x_star))
        if xTx > RAIO_DOE**2 + TOL_ESFERA_NBI:
            raise RuntimeError(
                f"Ótimo individual {objetivos_combo[idx_vertice]} fora da esfera: "
                f"xTx={xTx:.16g} > R²={RAIO_DOE**2:.16g}."
            )

        z_star = np.r_[x_star, 0.0]
        eq_vec = eq_nbi(z_star)
        eq_inf = float(np.max(np.abs(eq_vec)))
        eq_l2 = float(np.linalg.norm(eq_vec))
        esfera_viol = max(0.0, xTx - RAIO_DOE**2)

        y_fisico, y_min, y_scaled, _ = predizer_ymin_scaled_jac(x_star)
        # Conta também a reavaliação final usada para preencher a saída.
        n_avaliacoes_rsm = int(contador_rsm.n_avaliacoes)

        linha = {
            "combo_id": int(combo_id),
            "beta_id": int(beta_id),
            "k": int(k),
            "objetivos": tuple(objetivos_combo),
            "beta": tuple(beta),
            "q_chim": cache["q_chim"],
            "volume_chim": cache["volume_chim"],
            "success": True,
            "aceito": bool(eq_inf <= TOL_EQ and esfera_viol <= TOL_ESFERA_NBI),
            "status": 0,
            "message": "Vértice do simplex: âncora exata da payoff (sem SLSQP)",
            "t": 0.0,
            "eq_inf": eq_inf,
            "eq_l2": eq_l2,
            "xTx": xTx,
            "esfera_viol": esfera_viol,
            "endpoint_payoff": True,
            "objetivo_anchor": objetivos_combo[idx_vertice],
            "dist_anchor": 0.0,
            "start_escolhido": "payoff_anchor_exact",
            "n_tentativas": 0,
            "nit": 0,
            "nfev": 0,
            "njev": 0,
            "n_avaliacoes_rsm": n_avaliacoes_rsm,
            "fun": 0.0,
        }

        for c, v in zip(FACTOR_COLS, x_star):
            linha[c] = float(v)
        for c in RESPONSE_COLS:
            linha[f"{c}_fisico"] = float(y_fisico[c])
            linha[f"{c}_min"] = float(y_min[c])
            linha[f"{c}_scaled"] = float(y_scaled[c])

        return linha, z_star

    # --------------------------------------------------------
    # BETAS INTERNOS: warm start + interpolação das âncoras + retries
    # --------------------------------------------------------
    starts_locais = []
    nomes_starts = []
    chaves = set()

    def adicionar_start(nome, x, t=None):
        x = np.asarray(x, dtype=float).copy()
        norma = float(np.linalg.norm(x))
        if norma > RAIO_DOE:
            # Corrige somente arredondamento. Pontos realmente externos são
            # rejeitados, pois não devem entrar como chute do NBI.
            if norma <= RAIO_DOE * (1.0 + 1e-10):
                x *= (RAIO_DOE * (1.0 - 1e-12)) / norma
            else:
                return

        if t is None:
            t = _projetar_t_na_reta_nbi(x, phi, normal, idx_combo)
        z = np.r_[x, np.clip(float(t), bounds_z[-1][0], bounds_z[-1][1])]
        chave = _chave_start(z)
        if chave not in chaves:
            chaves.add(chave)
            starts_locais.append(z)
            nomes_starts.append(nome)

    # 1) Continuação ao longo da malha beta.
    if z0_warm is not None:
        z0_warm = np.asarray(z0_warm, dtype=float)
        adicionar_start("warm", z0_warm[:-1], z0_warm[-1])

    # 2) Combinação convexa das soluções individuais. Como a esfera é
    # convexa, este ponto é sempre viável quando as âncoras são viáveis.
    x_beta = beta @ X_anchor
    adicionar_start("barycentric_anchor", x_beta)

    # 3) Âncora do maior peso e demais âncoras, em ordem de relevância.
    for j in np.argsort(-beta):
        adicionar_start(f"anchor_{objetivos_combo[int(j)]}", X_anchor[int(j)])

    # 4) Centro da região experimental.
    adicionar_start("center", np.zeros(len(FACTOR_COLS)))

    candidatos = []

    def executar_start(nome_start, z0, tentativa):
        res = minimize(
            fun=objetivo,
            x0=z0,
            method="SLSQP",
            jac=jac_objetivo,
            bounds=bounds_z,
            constraints=constraints,
            options={
                "ftol": 1e-10,
                "maxiter": MAXITER,
                "disp": False,
            }
        )

        z_cand = np.asarray(res.x, dtype=float)
        x_cand = z_cand[:-1]
        eq_vec_cand = eq_nbi(z_cand)
        eq_inf_cand = float(np.max(np.abs(eq_vec_cand)))
        xTx_cand = float(np.dot(x_cand, x_cand))
        esfera_viol_cand = max(0.0, xTx_cand - RAIO_DOE**2)
        viavel = bool(
            eq_inf_cand <= TOL_EQ
            and esfera_viol_cand <= TOL_ESFERA_NBI
        )

        candidato = {
            "res": res,
            "z": z_cand,
            "eq_vec": eq_vec_cand,
            "eq_inf": eq_inf_cand,
            "xTx": xTx_cand,
            "esfera_viol": esfera_viol_cand,
            "viavel": viavel,
            "nome_start": nome_start,
            "tentativa": tentativa,
        }
        candidatos.append(candidato)
        return candidato

    # Starts principais: normalmente warm ou interpolação resolvem de primeira.
    convergiu = False
    tentativa = 0
    for nome_start, z0 in zip(nomes_starts, starts_locais):
        tentativa += 1
        cand = executar_start(nome_start, z0, tentativa)
        if cand["viavel"] and bool(cand["res"].success):
            convergiu = True
            break

    # Resgate determinístico gerado somente quando todos os starts principais
    # falham. Isso evita inflar o orçamento computacional em execuções normais.
    if not convergiu:
        rng = np.random.default_rng(1000003 + 1009 * int(combo_id) + int(beta_id))
        for r_id in range(8):
            v = rng.normal(size=len(FACTOR_COLS))
            v /= np.linalg.norm(v)
            r = RAIO_DOE * (rng.random() ** (1.0 / len(FACTOR_COLS)))
            x_rescue = r * v
            t_rescue = _projetar_t_na_reta_nbi(x_rescue, phi, normal, idx_combo)
            z_rescue = np.r_[x_rescue, np.clip(t_rescue, bounds_z[-1][0], bounds_z[-1][1])]
            tentativa += 1
            cand = executar_start(f"rescue_{r_id + 1}", z_rescue, tentativa)
            if cand["viavel"] and bool(cand["res"].success):
                break

    if not candidatos:
        raise RuntimeError("Nenhum start válido foi gerado para o subproblema NBI.")

    # Prioridade: viabilidade; depois maior t; em caso de inviabilidade,
    # menor violação normalizada das restrições.
    def chave_candidato(c):
        t_c = float(c["z"][-1])
        viol = c["eq_inf"] / max(TOL_EQ, 1e-16) + c["esfera_viol"] / max(TOL_ESFERA_NBI, 1e-16)
        if c["viavel"]:
            return (0, -t_c, c["eq_inf"], c["esfera_viol"])
        return (1, viol, -t_c, c["eq_inf"])

    escolhido = min(candidatos, key=chave_candidato)
    res = escolhido["res"]
    z_star = escolhido["z"]
    x_star = z_star[:-1]
    t_star = z_star[-1]
    eq_vec = escolhido["eq_vec"]
    eq_inf = escolhido["eq_inf"]
    eq_l2 = float(np.linalg.norm(eq_vec))
    esfera_viol = escolhido["esfera_viol"]

    y_fisico, y_min, y_scaled, _ = predizer_ymin_scaled_jac(x_star)
    # Conta também a reavaliação final usada para preencher a saída.
    n_avaliacoes_rsm = int(contador_rsm.n_avaliacoes)

    linha = {
        "combo_id": int(combo_id),
        "beta_id": int(beta_id),
        "k": int(k),
        "objetivos": tuple(objetivos_combo),
        "beta": tuple(beta),
        "q_chim": cache["q_chim"],
        "volume_chim": cache["volume_chim"],
        "success": bool(res.success),
        "aceito": bool(eq_inf <= TOL_EQ and esfera_viol <= TOL_ESFERA_NBI),
        "status": int(res.status),
        "message": str(res.message),
        "t": float(t_star),
        "eq_inf": eq_inf,
        "eq_l2": eq_l2,
        "xTx": float(np.dot(x_star, x_star)),
        "esfera_viol": float(esfera_viol),
        "endpoint_payoff": False,
        "objetivo_anchor": None,
        "dist_anchor": np.nan,
        "start_escolhido": escolhido["nome_start"],
        "n_tentativas": int(escolhido["tentativa"]),
        "nit": int(res.nit) if hasattr(res, "nit") else np.nan,
        "nfev": int(res.nfev) if hasattr(res, "nfev") else np.nan,
        "njev": int(res.njev) if hasattr(res, "njev") else np.nan,
        "n_avaliacoes_rsm": n_avaliacoes_rsm,
        "fun": float(res.fun),
    }

    for c, v in zip(FACTOR_COLS, x_star):
        linha[c] = float(v)

    for c in RESPONSE_COLS:
        linha[f"{c}_fisico"] = float(y_fisico[c])
        linha[f"{c}_min"] = float(y_min[c])
        linha[f"{c}_scaled"] = float(y_scaled[c])

    return linha, z_star


# ============================================================
# 6) RESOLVER UMA COMBINAÇÃO SEQUENCIALMENTE
# ============================================================

def resolver_combo_sequencial(combo_id):
    cache = combo_cache[int(combo_id)]
    k = cache["k"]

    pares_betas = ordenar_betas_vizinho_mais_proximo(betas_por_k[k])

    resultados = []
    z0_warm = None

    for beta_id, beta in pares_betas:
        try:
            linha, z_star = resolver_nbi_beta_cached(
                combo_id=combo_id,
                beta_id=beta_id,
                beta=beta,
                z0_warm=z0_warm
            )

            resultados.append(linha)

            if linha["aceito"]:
                z0_warm = z_star
            else:
                z0_warm = None

        except Exception as e:
            resultados.append({
                "combo_id": int(combo_id),
                "beta_id": int(beta_id),
                "k": int(k),
                "objetivos": cache["objetivos_combo"],
                "beta": tuple(beta),
                "q_chim": cache["q_chim"],
                "volume_chim": cache["volume_chim"],
                "success": False,
                "aceito": False,
                "message": str(e),
                "nit": np.nan,
                "nfev": np.nan,
                "njev": np.nan,
                "n_avaliacoes_rsm": int(contador_rsm.n_avaliacoes),
            })
            z0_warm = None

    return resultados


# ============================================================
# 7) CHECKPOINT
# ============================================================

if path_checkpoint.exists():
    df_checkpoint = pd.read_csv(path_checkpoint)
    if "aceito" in df_checkpoint.columns:
        df_checkpoint["aceito"] = df_checkpoint["aceito"].fillna(False).astype(bool)
    if "success" in df_checkpoint.columns:
        df_checkpoint["success"] = df_checkpoint["success"].fillna(False).astype(bool)
    linhas_resultados = df_checkpoint.to_dict("records")
    _n_linhas_checkpoint_inicial = len(linhas_resultados)
    combos_feitos = set(df_checkpoint["combo_id"].astype(int).unique())
    print(f"\nCheckpoint encontrado: {len(df_checkpoint)} linhas.")
else:
    linhas_resultados = []
    _n_linhas_checkpoint_inicial = 0
    combos_feitos = set()
    print("\nNenhum checkpoint encontrado.")

combo_ids_todos = list(combo_cache.keys())
combo_ids_restantes = [c for c in combo_ids_todos if c not in combos_feitos]

print(f"Combinações totais: {len(combo_ids_todos)}")
print(f"Combinações já feitas: {len(combos_feitos)}")
print(f"Combinações restantes: {len(combo_ids_restantes)}")

total_restante = 0
for cid in combo_ids_restantes:
    k = combo_cache[cid]["k"]
    total_restante += len(betas_por_k[k])

print(f"Subproblemas NBI restantes: {total_restante}")


# ============================================================
# 8) LOOP PARALELO EM BATCHES
# ============================================================

def chunks(lista, tamanho):
    for i in range(0, len(lista), tamanho):
        yield lista[i:i + tamanho]


batches = list(chunks(combo_ids_restantes, BATCH_SIZE_COMBOS))

t0_cnbi_loop = perf_counter()

iter_batches = tqdm(batches, desc="Batches CNBI") if tqdm is not None else batches

for b_id, batch in enumerate(iter_batches, start=1):

    print(f"\nBatch {b_id}/{len(batches)} | combos={len(batch)}")

    if JOBLIB_OK and N_JOBS > 1 and len(batch) > 1:
        saida_batch = Parallel(n_jobs=N_JOBS, backend=BACKEND, verbose=0)(
            delayed(resolver_combo_sequencial)(combo_id)
            for combo_id in batch
        )

        novos = []
        for lista in saida_batch:
            novos.extend(lista)
    else:
        novos = []
        for combo_id in batch:
            novos.extend(resolver_combo_sequencial(combo_id))

    linhas_resultados.extend(novos)

    df_temp = pd.DataFrame(linhas_resultados)
    df_temp.to_csv(path_checkpoint, index=False)

    taxa = df_temp["aceito"].mean() if "aceito" in df_temp.columns else np.nan
    tempo = (perf_counter() - t0_cnbi_loop) / 60

    print(
        f"Checkpoint salvo | linhas={len(df_temp)} | "
        f"aceitos={taxa:.2%} | tempo={tempo:.2f} min"
    )


# ============================================================
# 9) RESULTADOS FINAIS
# ============================================================

df_cnbi_raw = pd.DataFrame(linhas_resultados)
if "aceito" in df_cnbi_raw.columns:
    df_cnbi_raw["aceito"] = df_cnbi_raw["aceito"].fillna(False).astype(bool)
if "success" in df_cnbi_raw.columns:
    df_cnbi_raw["success"] = df_cnbi_raw["success"].fillna(False).astype(bool)
df_cnbi_raw.to_csv(OUTPUT_DIR / f"cnbi_raw_todas_mantidas_rsm_jac_{delta_tag}.csv", index=False)

df_cnbi_validas = df_cnbi_raw[
    (df_cnbi_raw["aceito"] == True) &
    (df_cnbi_raw["eq_inf"] <= TOL_EQ) &
    (df_cnbi_raw["xTx"] <= RAIO_DOE**2 + TOL_ESFERA_NBI)
].copy()

df_cnbi_validas.to_csv(
    OUTPUT_DIR / f"cnbi_validas_todas_mantidas_rsm_jac_{delta_tag}.csv",
    index=False
)

# Auditoria estrutural: a execução deve produzir exatamente um resultado por
# par (combinação, beta), totalizando 616 para os 26 sub-CHIMs selecionados.
n_esperado = int(sum(len(betas_por_k[combo_cache[cid]["k"]]) for cid in combo_cache))
chaves_repetidas = int(df_cnbi_raw.duplicated(["combo_id", "beta_id"]).sum())

if len(df_cnbi_raw) != n_esperado or chaves_repetidas != 0:
    raise RuntimeError(
        f"Estrutura incompleta do CNBI: linhas={len(df_cnbi_raw)}, "
        f"esperado={n_esperado}, duplicadas={chaves_repetidas}."
    )

# Nos vértices, o ponto deve reproduzir exatamente a âncora da payoff.
df_endpoints = df_cnbi_raw[df_cnbi_raw["endpoint_payoff"] == True].copy()
falhas_endpoint = df_endpoints[
    (df_endpoints["aceito"] != True)
    | (df_endpoints["eq_inf"] > TOL_EQ)
    | (df_endpoints["xTx"] > RAIO_DOE**2 + TOL_ESFERA_NBI)
    | (df_endpoints["dist_anchor"].abs() > 1e-12)
]

if not falhas_endpoint.empty:
    display(falhas_endpoint[[
        "combo_id", "beta_id", "objetivos", "beta", "objetivo_anchor",
        "eq_inf", "xTx", "dist_anchor", "message"
    ]])
    raise RuntimeError("Há vértices do simplex que não reproduziram a payoff.")

if len(df_cnbi_validas) != n_esperado:
    falhas = df_cnbi_raw[df_cnbi_raw["aceito"] != True].copy()
    display(falhas[[
        "combo_id", "beta_id", "objetivos", "beta", "success", "status",
        "eq_inf", "xTx", "esfera_viol", "start_escolhido", "n_tentativas", "message"
    ]])
    raise RuntimeError(
        f"CNBI ainda incompleto: {len(df_cnbi_validas)}/{n_esperado} válidos."
    )

print(f"AUDITORIA OK: {len(df_cnbi_validas)}/{n_esperado} subproblemas válidos.")
print(f"Vértices da payoff reproduzidos exatamente: {len(df_endpoints)}.")

# Tempo exclusivo do loop CNBI na sessão atual. Mantemos a grandeza
# básica em segundos e derivamos os minutos apenas para exibição.
tempo_total_loop_s = float(perf_counter() - t0_cnbi_loop)
tempo_total = tempo_total_loop_s / 60.0

print("\n" + "=" * 80)
print("CNBI finalizado")
print("=" * 80)
print(f"Tempo desta execução: {tempo_total:.2f} min")
print(f"Soluções brutas: {len(df_cnbi_raw)}")
print(f"Soluções válidas: {len(df_cnbi_validas)}")
print(f"Taxa válida: {len(df_cnbi_validas) / max(len(df_cnbi_raw), 1):.2%}")

# ------------------------------------------------------------
# ORÇAMENTO COMPUTACIONAL TOTAL (avaliações do RSM)
# ------------------------------------------------------------
# Moeda de orçamento adotada: número de avaliações do modelo RSM
# (contador próprio, agnóstico ao solver — ver Seção 2b desta célula).
# Inclui TODOS os subproblemas rodados nesta execução, inclusive os
# rejeitados (aceito=False), pois eles também consumiram avaliações.
# Este bloco mantém o subtotal das avaliações do RSM consumidas pelo CNBI.
# O orçamento consolidado, criado após o pós-processamento, adiciona payoff,
# Monte Carlo, SVDs, pré-seleção geométrica e pós-processamento.
orcamento_cnbi_total = int(df_cnbi_raw["n_avaliacoes_rsm"].sum())
orcamento_cnbi_validas = int(df_cnbi_raw.loc[df_cnbi_raw["aceito"], "n_avaliacoes_rsm"].sum())

_mask_sessao_cnbi = df_cnbi_raw["combo_id"].astype(int).isin(
    set(int(c) for c in combo_ids_restantes)
)
orcamento_cnbi_sessao = int(
    df_cnbi_raw.loc[_mask_sessao_cnbi, "n_avaliacoes_rsm"].sum()
)
n_subproblemas_sessao = int(_mask_sessao_cnbi.sum())
checkpoint_reutilizado = bool(_n_linhas_checkpoint_inicial > 0)

_reg_cnbi = registrar_custo_etapa(
    chave="05_otimizacao_cnbi",
    ordem=50,
    categoria="otimizacao_cnbi",
    timer=_timer_cnbi,
    # Total acumulado é a moeda usada para comparar o método completo.
    avaliacoes_rsm=orcamento_cnbi_total,
    n_svd=len(combo_cache),
    n_subproblemas=len(df_cnbi_raw),
    n_combinacoes=len(combo_cache),
    avaliacoes_rsm_sessao=orcamento_cnbi_sessao,
    n_subproblemas_sessao=n_subproblemas_sessao,
    n_subproblemas_checkpoint_inicial=int(_n_linhas_checkpoint_inicial),
    checkpoint_reutilizado=checkpoint_reutilizado,
    tempo_parede_loop_cnbi_s=float(tempo_total_loop_s),
    n_subproblemas_aceitos=int(df_cnbi_raw["aceito"].sum()),
    observacao=(
        "Inclui a montagem do cache local, uma SVD por sub-CHIM selecionado, "
        "a solução dos subproblemas e a auditoria. Quando há checkpoint, as "
        "avaliações são acumuladas, mas o tempo de parede corresponde apenas "
        "à sessão atual."
    ),
)

print("\n" + "-" * 80)
print("ORÇAMENTO COMPUTACIONAL (avaliações do RSM)")
print("-" * 80)
print(f"Avaliações do RSM — total (todos os subproblemas rodados): {orcamento_cnbi_total}")
print(f"Avaliações do RSM — apenas subproblemas aceitos:           {orcamento_cnbi_validas}")
print(f"Avaliações do RSM — executadas nesta sessão:               {orcamento_cnbi_sessao}")
print(f"Tempo de parede da etapa CNBI nesta sessão:                {_reg_cnbi['tempo_parede_s']:.3f} s")
if checkpoint_reutilizado:
    print("AVISO: checkpoint reutilizado; o tempo de parede não representa a execução acumulada completa.")
print(
    f"Avaliações por subproblema (média): "
    f"{df_cnbi_raw['n_avaliacoes_rsm'].mean():.2f} "
    f"(mediana: {df_cnbi_raw['n_avaliacoes_rsm'].median():.1f}, "
    f"min: {df_cnbi_raw['n_avaliacoes_rsm'].min():.0f}, "
    f"max: {df_cnbi_raw['n_avaliacoes_rsm'].max():.0f})"
)
print(
    "\nApós o pós-processamento, use ORCAMENTO_RSM_TOTAL_METODO para "
    "configurar NSGA-III / MOEA/D com payoff + CNBI no mesmo orçamento "
    "de avaliações do RSM."
)

df_orcamento_cnbi = pd.DataFrame([{
    "avaliacoes_rsm_total_acumuladas": orcamento_cnbi_total,
    "avaliacoes_rsm_validas": orcamento_cnbi_validas,
    "avaliacoes_rsm_sessao": orcamento_cnbi_sessao,
    "n_subproblemas_rodados": len(df_cnbi_raw),
    "n_subproblemas_aceitos": int(df_cnbi_raw["aceito"].sum()),
    "n_subproblemas_sessao": n_subproblemas_sessao,
    "n_subproblemas_checkpoint_inicial": int(_n_linhas_checkpoint_inicial),
    "checkpoint_reutilizado": checkpoint_reutilizado,
    "n_svd_cache_subchim": len(combo_cache),
    "tempo_parede_etapa_sessao_s": float(_reg_cnbi["tempo_parede_s"]),
    "tempo_parede_loop_sessao_s": float(tempo_total_loop_s),
    "avaliacoes_media_por_subproblema": float(df_cnbi_raw["n_avaliacoes_rsm"].mean()),
    "avaliacoes_mediana_por_subproblema": float(df_cnbi_raw["n_avaliacoes_rsm"].median()),
}])
display(df_orcamento_cnbi)

display(
    df_cnbi_validas
    .groupby("k")
    .agg(
        n=("aceito", "size"),
        t_mediano=("t", "median"),
        eq_inf_mediano=("eq_inf", "median"),
        q_mediano=("q_chim", "median"),
        volume_mediano=("volume_chim", "median")
    )
)

display(df_cnbi_validas.head())

In [ ]:
# ============================================================
# DIAGNÓSTICO CNBI POR k
# ============================================================

df_diag_k = (
    df_cnbi_raw
    .groupby("k")
    .agg(
        brutas=("combo_id", "size"),
        aceitas=("aceito", "sum"),
        taxa_aceita=("aceito", "mean"),
        eq_inf_mediana=("eq_inf", "median"),
        eq_inf_p90=("eq_inf", lambda x: x.quantile(0.90)),
        eq_inf_max=("eq_inf", "max"),
        t_mediano=("t", "median"),
        q_mediano=("q_chim", "median"),
        volume_mediano=("volume_chim", "median"),
        nit_mediana=("nit", "median")
    )
    .reset_index()
)

display(
    df_diag_k.style.format({
        "taxa_aceita": "{:.2%}",
        "eq_inf_mediana": "{:.2e}",
        "eq_inf_p90": "{:.2e}",
        "eq_inf_max": "{:.2e}",
        "t_mediano": "{:.4f}",
        "q_mediano": "{:.3f}",
        "volume_mediano": "{:.6g}",
        "nit_mediana": "{:.1f}"
    })
)

In [ ]:
# ============================================================
# PIORES COMBINAÇÕES POR TAXA DE ACEITAÇÃO
# ============================================================

df_diag_combo = (
    df_cnbi_raw
    .groupby(["combo_id", "k", "objetivos"])
    .agg(
        brutas=("combo_id", "size"),
        aceitas=("aceito", "sum"),
        taxa_aceita=("aceito", "mean"),
        q_chim=("q_chim", "first"),
        volume_chim=("volume_chim", "first"),
        eq_inf_mediana=("eq_inf", "median"),
        eq_inf_max=("eq_inf", "max"),
        nit_mediana=("nit", "median")
    )
    .reset_index()
    .sort_values(["taxa_aceita", "eq_inf_mediana"], ascending=[True, False])
)

display(
    df_diag_combo.head(30).style.format({
        "taxa_aceita": "{:.2%}",
        "q_chim": "{:.3f}",
        "volume_chim": "{:.6g}",
        "eq_inf_mediana": "{:.2e}",
        "eq_inf_max": "{:.2e}",
        "nit_mediana": "{:.1f}"
    })
)

In [ ]:
_timer_pos_duplicados = iniciar_medicao_etapa()

# ============================================================
# FILTRO: REMOVER X DUPLICADOS + MANTER NÃO DOMINADOS
# E contar sobreviventes por k
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1) Configurações
# ------------------------------------------------------------

X_COLS = FACTOR_COLS
F_COLS = [f"{c}_min" for c in RESPONSE_COLS]  # usado no filtro de dominância
F_COLS_SCORE = [f"{c}_scaled" for c in RESPONSE_COLS]  # usado para desempatar X duplicado

TOL_X = 1e-5
TOL_DOM = 1e-10

df_base = df_cnbi_validas.copy()

# ------------------------------------------------------------
# 2) Remover duplicados em X
#    Arredonda X para agrupar pontos praticamente iguais
# ------------------------------------------------------------

def remover_x_duplicados(df, x_cols, f_cols, tol_x=1e-5):
    df = df.copy()

    casas = int(np.ceil(-np.log10(tol_x)))

    for c in x_cols:
        df[f"{c}_round"] = df[c].round(casas)

    x_round_cols = [f"{c}_round" for c in x_cols]

    # Se há X duplicado, mantém o ponto com menor soma dos objetivos minimizados
    score_cols = F_COLS_SCORE if all(c in df.columns for c in F_COLS_SCORE) else f_cols
    df["_score_min"] = df[score_cols].sum(axis=1)

    df_sem_dup = (
        df.sort_values("_score_min", ascending=True)
          .drop_duplicates(subset=x_round_cols, keep="first")
          .drop(columns=x_round_cols + ["_score_min"])
          .reset_index(drop=True)
    )

    return df_sem_dup


df_cnbi_sem_dup_x = remover_x_duplicados(
    df_base,
    x_cols=X_COLS,
    f_cols=F_COLS,
    tol_x=TOL_X
)

print("Antes:", len(df_base))
print("Depois de remover X duplicados:", len(df_cnbi_sem_dup_x))

registrar_custo_etapa(
    chave="06a_pos_remocao_duplicados",
    ordem=60,
    categoria="pos_processamento",
    timer=_timer_pos_duplicados,
    n_subproblemas=len(df_base),
    n_pontos_entrada=len(df_base),
    n_pontos_saida=len(df_cnbi_sem_dup_x),
    n_pontos_removidos=len(df_base) - len(df_cnbi_sem_dup_x),
    observacao="Agrupamento por coordenadas de decisão arredondadas e desempate por escore.",
)


In [ ]:
_timer_pos_dominancia = iniciar_medicao_etapa()

# ------------------------------------------------------------
# 3) Filtro de não dominados
#    Minimização em todos os objetivos
# ------------------------------------------------------------

def mascara_nao_dominados(F, tol=1e-10):
    """
    Retorna máscara True para pontos não dominados.
    Assume minimização.

    Ponto i é dominado se existe ponto j tal que:
        F_j <= F_i em todos os objetivos
        F_j <  F_i em pelo menos um objetivo
    """

    F = np.asarray(F, dtype=float)
    n = F.shape[0]

    is_nd = np.ones(n, dtype=bool)

    for i in range(n):
        if not is_nd[i]:
            continue

        # Quem domina i?
        domina_i = (
            np.all(F <= F[i] + tol, axis=1) &
            np.any(F < F[i] - tol, axis=1)
        )

        domina_i[i] = False

        if np.any(domina_i):
            is_nd[i] = False

    return is_nd


F = df_cnbi_sem_dup_x[F_COLS].values

mask_nd = mascara_nao_dominados(F, tol=TOL_DOM)

df_cnbi_nd = df_cnbi_sem_dup_x[mask_nd].copy().reset_index(drop=True)

print("Depois de remover dominados:", len(df_cnbi_nd))

registrar_custo_etapa(
    chave="06b_pos_filtro_dominancia",
    ordem=61,
    categoria="pos_processamento",
    timer=_timer_pos_dominancia,
    n_subproblemas=len(df_cnbi_sem_dup_x),
    n_pontos_entrada=len(df_cnbi_sem_dup_x),
    n_pontos_saida=len(df_cnbi_nd),
    comparacoes_dominancia_max=int(len(df_cnbi_sem_dup_x) ** 2),
    observacao="Filtro exato de não dominância em todos os objetivos minimizados.",
)


In [ ]:
_timer_pos_resumo_k = iniciar_medicao_etapa()

# ------------------------------------------------------------
# 4) Quantos sobrevivem a cada etapa por k
# ------------------------------------------------------------

resumo_original = (
    df_base.groupby("k")
    .size()
    .rename("validas_iniciais")
)

resumo_sem_dup = (
    df_cnbi_sem_dup_x.groupby("k")
    .size()
    .rename("apos_remover_x_duplicado")
)

resumo_nd = (
    df_cnbi_nd.groupby("k")
    .size()
    .rename("nao_dominadas")
)

df_resumo_filtros_k = pd.concat(
    [resumo_original, resumo_sem_dup, resumo_nd],
    axis=1
).fillna(0).astype(int).reset_index()

df_resumo_filtros_k["removidas_por_x_duplicado"] = (
    df_resumo_filtros_k["validas_iniciais"]
    - df_resumo_filtros_k["apos_remover_x_duplicado"]
)

df_resumo_filtros_k["dominadas_removidas"] = (
    df_resumo_filtros_k["apos_remover_x_duplicado"]
    - df_resumo_filtros_k["nao_dominadas"]
)

df_resumo_filtros_k["perc_sobrevive_x"] = (
    df_resumo_filtros_k["apos_remover_x_duplicado"]
    / df_resumo_filtros_k["validas_iniciais"]
)

df_resumo_filtros_k["perc_sobrevive_nd"] = (
    df_resumo_filtros_k["nao_dominadas"]
    / df_resumo_filtros_k["validas_iniciais"]
)

display(
    df_resumo_filtros_k.style.format({
        "perc_sobrevive_x": "{:.2%}",
        "perc_sobrevive_nd": "{:.2%}",
    })
)

registrar_custo_etapa(
    chave="06c_pos_resumo_por_k",
    ordem=62,
    categoria="pos_processamento",
    timer=_timer_pos_resumo_k,
    n_subproblemas=len(df_base),
    observacao="Consolidação dos sobreviventes e remoções por cardinalidade k.",
)


In [ ]:
_timer_pos_resumo_final = iniciar_medicao_etapa()

# ============================================================
# RESUMO FINAL DOS FILTROS
# ============================================================

total_inicial = df_resumo_filtros_k["validas_iniciais"].sum()
total_sem_dup = df_resumo_filtros_k["apos_remover_x_duplicado"].sum()
total_nd = df_resumo_filtros_k["nao_dominadas"].sum()

print("Total válidas iniciais:", total_inicial)
print("Após remover X duplicados:", total_sem_dup)
print("Não dominadas finais:", total_nd)
print("Sobrevivência final:", total_nd / total_inicial)

display(
    df_resumo_filtros_k.style.format({
        "perc_sobrevive_x": "{:.2%}",
        "perc_sobrevive_nd": "{:.2%}",
    })
)

registrar_custo_etapa(
    chave="06d_pos_resumo_final",
    ordem=63,
    categoria="pos_processamento",
    timer=_timer_pos_resumo_final,
    n_subproblemas=int(total_inicial),
    n_pontos_finais=int(total_nd),
    observacao="Resumo final da sobrevivência após duplicidade e dominância.",
)


In [ ]:
_timer_pos_exportacao = iniciar_medicao_etapa()

# ============================================================
# EXPORTAÇÃO DA FRONTEIRA CNBI FINAL
# Salva CSV e Excel com a fronteira não dominada e diagnósticos.
# ============================================================

from pathlib import Path
import json

EXPORT_DIR = OUTPUT_DIR / "fronteira_final"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

TAG_FRONTEIRA = (
    f"cnbi_{RSM_MODELO}"
    f"_d{D_CHIM_REFERENCIA}"
    f"_piso{str(round(SIGMA_PISO_RESIDUAL, 8)).replace('.', 'p')}"
)

arquivo_csv_nd = EXPORT_DIR / f"fronteira_ND_{TAG_FRONTEIRA}.csv"
arquivo_csv_sem_dup = EXPORT_DIR / f"fronteira_apos_X_duplicado_{TAG_FRONTEIRA}.csv"
arquivo_excel = EXPORT_DIR / f"fronteira_CNBI_{TAG_FRONTEIRA}.xlsx"


def preparar_para_exportacao(df_export):
    """
    Converte listas/tuplas/dicts para texto para evitar problemas no Excel/CSV.
    Mantém numéricos como numéricos.
    """
    df_export = df_export.copy()

    for col in df_export.columns:
        if df_export[col].dtype == "object":
            df_export[col] = df_export[col].map(
                lambda v: json.dumps(v, ensure_ascii=False)
                if isinstance(v, (list, tuple, dict))
                else v
            )

    return df_export


abas_excel = {
    "fronteira_ND": df_cnbi_nd,
    "apos_X_duplicado": df_cnbi_sem_dup_x,
    "validas_pre_filtros": df_cnbi_validas,
    "resumo_filtros_k": df_resumo_filtros_k,
    "diagnostico_k": df_diag_k,
    "diagnostico_combo": df_diag_combo,
    "janela_singular": df_resumo_janela_singular,
    "orcamento_cnbi": df_orcamento_cnbi,
    "sub_CHIMs_aceitos": df_combinacoes_filtradas,
    "sub_CHIMs_todos": df_combinacoes_com_q,
}

# CSVs principais.
preparar_para_exportacao(df_cnbi_nd).to_csv(
    arquivo_csv_nd,
    index=False,
    encoding="utf-8-sig"
)

preparar_para_exportacao(df_cnbi_sem_dup_x).to_csv(
    arquivo_csv_sem_dup,
    index=False,
    encoding="utf-8-sig"
)

# Excel consolidado com múltiplas abas.
with pd.ExcelWriter(arquivo_excel, engine="openpyxl") as writer:
    for nome_aba, df_aba in abas_excel.items():
        nome_aba_excel = nome_aba[:31]
        preparar_para_exportacao(df_aba).to_excel(
            writer,
            sheet_name=nome_aba_excel,
            index=False
        )

print("=" * 70)
print("Fronteira CNBI exportada")
print("=" * 70)
print(f"CSV fronteira ND:           {arquivo_csv_nd.resolve()}")
print(f"CSV após X duplicado:       {arquivo_csv_sem_dup.resolve()}")
print(f"Excel consolidado:          {arquivo_excel.resolve()}")
print(f"n fronteira ND:             {len(df_cnbi_nd)}")
print(f"n após remover X duplicado: {len(df_cnbi_sem_dup_x)}")
print(f"n válidas pré-filtros:      {len(df_cnbi_validas)}")

display(df_resumo_janela_singular)
display(df_resumo_filtros_k)


registrar_custo_etapa(
    chave="06e_pos_exportacao",
    ordem=64,
    categoria="pos_processamento",
    timer=_timer_pos_exportacao,
    n_subproblemas=len(df_cnbi_nd),
    n_arquivos_exportados=3,
    observacao="Exportação da fronteira, diagnósticos e tabelas consolidadas.",
)


In [ ]:
# ============================================================
# ORÇAMENTO COMPUTACIONAL CONSOLIDADO DO MÉTODO
# ============================================================
# Reúne custos fixos e variáveis:
# payoff, análise paralela Monte Carlo, SVDs, pré-seleção geométrica,
# otimização CNBI e pós-processamento.

_colunas_base_orcamento = [
    "chave", "ordem", "categoria",
    "avaliacoes_rsm", "n_svd", "n_mc",
    "n_subproblemas", "n_combinacoes",
    "tempo_parede_s", "tempo_cpu_processo_principal_s",
    "observacao",
]

df_orcamento_etapas = (
    pd.DataFrame(list(REGISTRO_CUSTO_PIPELINE.values()))
    .sort_values(["ordem", "chave"])
    .reset_index(drop=True)
)

for _c in _colunas_base_orcamento:
    if _c not in df_orcamento_etapas.columns:
        df_orcamento_etapas[_c] = 0 if _c not in {"chave", "categoria", "observacao"} else ""

# Totais nas moedas que são aditivas entre etapas.
_colunas_soma = [
    "avaliacoes_rsm",
    "n_svd",
    "n_mc",
    "n_subproblemas",
    "n_combinacoes",
    "tempo_parede_s",
    "tempo_cpu_processo_principal_s",
]

df_orcamento_categorias = (
    df_orcamento_etapas
    .groupby("categoria", as_index=False)[_colunas_soma]
    .sum()
)

ORCAMENTO_RSM_TOTAL_METODO = int(df_orcamento_etapas["avaliacoes_rsm"].sum())
N_SVD_TOTAL_METODO = int(df_orcamento_etapas["n_svd"].sum())
N_MC_TOTAL_METODO = int(df_orcamento_etapas["n_mc"].sum())

# Soma dos cronômetros locais: não incorpora pausas entre células e é a
# referência recomendada para execução manual ou "Run All".
TEMPO_PAREDE_TOTAL_METODO_S = float(df_orcamento_etapas["tempo_parede_s"].sum())

# Diagnóstico adicional. Só representa um tempo de pipeline contínuo quando
# todas as células são executadas com "Run All" e sem pausas.
TEMPO_DESDE_IMPORTACOES_S = float(perf_counter() - PIPELINE_WALL_T0)

avaliacoes_payoff = int(
    df_orcamento_etapas.loc[
        df_orcamento_etapas["categoria"] == "construcao_payoff",
        "avaliacoes_rsm",
    ].sum()
)
avaliacoes_cnbi = int(
    df_orcamento_etapas.loc[
        df_orcamento_etapas["categoria"] == "otimizacao_cnbi",
        "avaliacoes_rsm",
    ].sum()
)

_checkpoint_reutilizado_orcamento = bool(
    df_orcamento_etapas.get("checkpoint_reutilizado", pd.Series(dtype=bool))
    .fillna(False)
    .astype(bool)
    .any()
)

df_orcamento_resumo = pd.DataFrame([{
    "avaliacoes_rsm_payoff": avaliacoes_payoff,
    "avaliacoes_rsm_cnbi": avaliacoes_cnbi,
    "avaliacoes_rsm_total_metodo": ORCAMENTO_RSM_TOTAL_METODO,
    "n_svd_total": N_SVD_TOTAL_METODO,
    "n_replicas_mc_total": N_MC_TOTAL_METODO,
    "tempo_parede_total_etapas_s": TEMPO_PAREDE_TOTAL_METODO_S,
    "tempo_parede_total_etapas_min": TEMPO_PAREDE_TOTAL_METODO_S / 60.0,
    "tempo_desde_importacoes_s": TEMPO_DESDE_IMPORTACOES_S,
    "checkpoint_reutilizado_cnbi": _checkpoint_reutilizado_orcamento,
    "tempo_parede_cnbi_completo": not _checkpoint_reutilizado_orcamento,
}])

print("=" * 90)
print("ORÇAMENTO COMPUTACIONAL CONSOLIDADO")
print("=" * 90)
print(f"Avaliações do RSM — payoff:        {avaliacoes_payoff}")
print(f"Avaliações do RSM — CNBI:          {avaliacoes_cnbi}")
print(f"Avaliações do RSM — método total:  {ORCAMENTO_RSM_TOTAL_METODO}")
print(f"SVDs explícitas — método total:     {N_SVD_TOTAL_METODO}")
print(f"Réplicas Monte Carlo — total:       {N_MC_TOTAL_METODO}")
print(
    "Tempo de parede somado das etapas: "
    f"{TEMPO_PAREDE_TOTAL_METODO_S:.3f} s "
    f"({TEMPO_PAREDE_TOTAL_METODO_S / 60.0:.3f} min)"
)

if _checkpoint_reutilizado_orcamento:
    print(
        "AVISO: o CNBI reutilizou checkpoint. As avaliações acumuladas estão "
        "corretas, mas o tempo de parede do CNBI cobre somente esta sessão. "
        "Para medir o pipeline completo, remova o checkpoint e execute Run All."
    )

display(
    df_orcamento_etapas.style.format({
        "tempo_parede_s": "{:.6f}",
        "tempo_cpu_processo_principal_s": "{:.6f}",
    })
)
display(
    df_orcamento_categorias.style.format({
        "tempo_parede_s": "{:.6f}",
        "tempo_cpu_processo_principal_s": "{:.6f}",
    })
)
display(df_orcamento_resumo)

# ------------------------------------------------------------
# Exportação do orçamento
# ------------------------------------------------------------
arquivo_orcamento_csv = OUTPUT_DIR / "orcamento_computacional_etapas.csv"
arquivo_orcamento_resumo_csv = OUTPUT_DIR / "orcamento_computacional_resumo.csv"
arquivo_orcamento_excel = OUTPUT_DIR / "orcamento_computacional_completo.xlsx"

df_orcamento_etapas.to_csv(arquivo_orcamento_csv, index=False, encoding="utf-8-sig")
df_orcamento_resumo.to_csv(
    arquivo_orcamento_resumo_csv, index=False, encoding="utf-8-sig"
)

with pd.ExcelWriter(arquivo_orcamento_excel, engine="openpyxl") as writer:
    preparar_para_exportacao(df_orcamento_etapas).to_excel(
        writer, sheet_name="etapas", index=False
    )
    preparar_para_exportacao(df_orcamento_categorias).to_excel(
        writer, sheet_name="categorias", index=False
    )
    preparar_para_exportacao(df_orcamento_resumo).to_excel(
        writer, sheet_name="resumo", index=False
    )
    preparar_para_exportacao(df_orcamento_cnbi).to_excel(
        writer, sheet_name="cnbi_detalhado", index=False
    )

# Também adiciona/substitui as abas de orçamento no Excel da fronteira.
if "arquivo_excel" in globals() and Path(arquivo_excel).exists():
    with pd.ExcelWriter(
        arquivo_excel,
        engine="openpyxl",
        mode="a",
        if_sheet_exists="replace",
    ) as writer:
        preparar_para_exportacao(df_orcamento_etapas).to_excel(
            writer, sheet_name="orcamento_etapas", index=False
        )
        preparar_para_exportacao(df_orcamento_resumo).to_excel(
            writer, sheet_name="orcamento_resumo", index=False
        )

print("Arquivos do orçamento:")
print(" -", arquivo_orcamento_csv.resolve())
print(" -", arquivo_orcamento_resumo_csv.resolve())
print(" -", arquivo_orcamento_excel.resolve())


## Comparação sob orçamento igual — NSGA-III e MOEA/D (múltiplas seeds)

Executa NSGA-III e MOEA/D **diretamente sobre os 8 objetivos originais**
(não a decomposição combinatória do CNBI), usando o mesmo modelo RSM e o
mesmo domínio esférico ($\|x\| \le RAIO\_DOE$). O orçamento (número de
avaliações do modelo RSM) é fixado em `orcamento_cnbi_total`.

**Tratamento da restrição esférica**: o MOEA/D do `pymoo` não aceita
restrições (`n_ieq_constr`). Por isso, NSGA-III usa a restrição explícita
(via `G`), enquanto o MOEA/D usa uma estratégia de **reparo por projeção**:
qualquer ponto fora da esfera é projetado de volta à sua superfície
($x \leftarrow x \cdot \min(1,\ RAIO\_DOE/\|x\|)$) antes de ser avaliado —
garante viabilidade sem exigir suporte nativo a restrições.

**Múltiplas seeds**: como NSGA-III e MOEA/D são estocásticos (CNBI não é),
cada algoritmo roda `N_SEEDS` vezes independentes com o mesmo orçamento,
para permitir comparação estatística (mediana/IQR) contra o CNBI, em vez de
um único ponto por algoritmo. Requer `pymoo` (`pip install -U pymoo`).


In [ ]:
# ============================================================
# NSGA-III E MOEA/D SOB O MESMO ORÇAMENTO DO CNBI — MÚLTIPLAS SEEDS
# ============================================================

try:
    from pymoo.core.problem import Problem
    from pymoo.algorithms.moo.nsga3 import NSGA3
    from pymoo.algorithms.moo.moead import MOEAD
    from pymoo.util.ref_dirs import get_reference_directions
    from pymoo.optimize import minimize as pymoo_minimize
    from pymoo.termination import get_termination
    PYMOO_OK = True
except ImportError:
    PYMOO_OK = False
    print("pymoo não está instalado. Rode: pip install -U pymoo")

if PYMOO_OK:

    # Orçamento de avaliações do método completo: payoff + CNBI.
    # SVD, Monte Carlo e pós-processamento permanecem reportados em suas
    # próprias moedas e pelo tempo de parede, sem conversão arbitrária.
    orcamento_referencia_rsm = int(
        globals().get("ORCAMENTO_RSM_TOTAL_METODO", orcamento_cnbi_total)
    )
    tempo_referencia_parede_s = float(
        globals().get("TEMPO_PAREDE_TOTAL_METODO_S", np.nan)
    )

    # -----------------------------
    # Preset: número de execuções independentes por algoritmo
    # -----------------------------
    N_SEEDS = 10  # ajuste conforme orçamento de tempo disponível
    SEEDS = list(range(1, N_SEEDS + 1))

    # ------------------------------------------------------------
    # 1) Avaliação vetorizada comum (reaproveita build_design_matrix/B_np)
    # ------------------------------------------------------------
    def avaliar_rsm_fisico(X):
        df_x = pd.DataFrame(X, columns=FACTOR_COLS)
        Xd = build_design_matrix(df_x, FACTOR_COLS, modelo=RSM_MODELO)
        return Xd.values @ B_np  # (n_pop, n_obj), unidades físicas

    def projetar_na_esfera(X, raio=RAIO_DOE):
        normas = np.linalg.norm(X, axis=1, keepdims=True)
        normas_seguras = np.where(normas == 0, 1.0, normas)
        fator = np.minimum(1.0, raio / normas_seguras)
        return X * fator

    # ------------------------------------------------------------
    # 2) Problema PARA NSGA-III — com restrição esférica explícita
    # ------------------------------------------------------------
    class ProblemaRSM_Restrito(Problem):
        def __init__(self):
            super().__init__(
                n_var=len(FACTOR_COLS),
                n_obj=len(RESPONSE_COLS),
                n_ieq_constr=1,
                xl=np.array([-RAIO_DOE] * len(FACTOR_COLS)),
                xu=np.array([RAIO_DOE] * len(FACTOR_COLS)),
            )
            self.n_avaliacoes_rsm = 0

        def _evaluate(self, X, out, *args, **kwargs):
            self.n_avaliacoes_rsm += X.shape[0]
            Y_fisico = avaliar_rsm_fisico(X)
            out["F"] = Y_fisico * SIGN
            out["G"] = np.sum(X ** 2, axis=1) - RAIO_DOE ** 2

    # ------------------------------------------------------------
    # 3) Problema PARA MOEA/D — sem restrições; reparo por projeção
    #    (MOEA/D do pymoo não suporta n_ieq_constr > 0)
    # ------------------------------------------------------------
    class ProblemaRSM_Reparo(Problem):
        def __init__(self):
            super().__init__(
                n_var=len(FACTOR_COLS),
                n_obj=len(RESPONSE_COLS),
                n_ieq_constr=0,
                xl=np.array([-RAIO_DOE] * len(FACTOR_COLS)),
                xu=np.array([RAIO_DOE] * len(FACTOR_COLS)),
            )
            self.n_avaliacoes_rsm = 0

        def _evaluate(self, X, out, *args, **kwargs):
            Xp = projetar_na_esfera(X, RAIO_DOE)
            self.n_avaliacoes_rsm += Xp.shape[0]
            Y_fisico = avaliar_rsm_fisico(Xp)
            out["F"] = Y_fisico * SIGN

    # ------------------------------------------------------------
    # 4) Direções de referência (Das-Dennis) — mesma malha para os dois
    # ------------------------------------------------------------
    N_PARTITIONS_REFDIRS = 3  # ajuste se quiser outro pop_size
    ref_dirs = get_reference_directions(
        "das-dennis", len(RESPONSE_COLS), n_partitions=N_PARTITIONS_REFDIRS
    )
    pop_size = len(ref_dirs)

    # ------------------------------------------------------------
    # 5) Orçamento: pop_size x n_gen ~= payoff + CNBI (por seed)
    # ------------------------------------------------------------
    n_gen = max(1, round(orcamento_referencia_rsm / pop_size))
    orcamento_efetivo = pop_size * n_gen
    termination = get_termination("n_gen", n_gen)

    print("=" * 70)
    print("Orçamento de avaliações igualado ao método completo (aplicado a CADA seed)")
    print("=" * 70)
    print(f"Orçamento do método completo (payoff + CNBI, avaliações do RSM): {orcamento_referencia_rsm}")
    print(f"Tempo de parede de referência do método: {tempo_referencia_parede_s:.3f} s")
    print(f"Direções de referência (Das-Dennis, p={N_PARTITIONS_REFDIRS}): {pop_size}")
    print(f"pop_size = {pop_size}  |  n_gen = {n_gen}  |  N_SEEDS = {N_SEEDS}")
    print(f"Orçamento efetivo por seed (pop_size x n_gen): {orcamento_efetivo}")
    print(
        f"Diferença em relação ao orçamento do método: "
        f"{(orcamento_efetivo - orcamento_referencia_rsm) / orcamento_referencia_rsm:+.2%}"
    )

    # ------------------------------------------------------------
    # 6) Loop de execução: N_SEEDS réplicas de cada algoritmo
    # ------------------------------------------------------------
    def exportar_fronteira_pymoo(F_min, X, nome_arquivo):
        Y_fisico = F_min * SIGN
        df_out = pd.DataFrame(X, columns=FACTOR_COLS)
        for i, c in enumerate(RESPONSE_COLS):
            df_out[c] = Y_fisico[:, i]
        caminho = EXPORT_DIR_EAS / nome_arquivo
        df_out.to_csv(caminho, index=False)
        return caminho

    EXPORT_DIR_EAS = OUTPUT_DIR / "fronteiras_ea_mesmo_orcamento"
    EXPORT_DIR_EAS.mkdir(parents=True, exist_ok=True)

    linhas_resumo = []

    for seed in tqdm(SEEDS, desc="Seeds") if tqdm is not None else SEEDS:

        # --- NSGA-III ---
        prob_nsga3 = ProblemaRSM_Restrito()
        t0 = perf_counter()
        res_nsga3 = pymoo_minimize(
            prob_nsga3,
            NSGA3(pop_size=pop_size, ref_dirs=ref_dirs),
            termination,
            seed=seed, save_history=False, verbose=False,
        )
        tempo_nsga3 = perf_counter() - t0

        caminho_nsga3 = exportar_fronteira_pymoo(
            res_nsga3.F, res_nsga3.X, f"nsga3_seed{seed:03d}.csv"
        )

        linhas_resumo.append({
            "metodo": "NSGA-III", "seed": seed,
            "avaliacoes_rsm": prob_nsga3.n_avaliacoes_rsm,
            "n_solucoes": len(res_nsga3.F),
            "tempo_s": tempo_nsga3,
            "arquivo": str(caminho_nsga3),
        })

        # --- MOEA/D ---
        prob_moead = ProblemaRSM_Reparo()
        t0 = perf_counter()
        res_moead = pymoo_minimize(
            prob_moead,
            MOEAD(ref_dirs=ref_dirs, n_neighbors=min(20, pop_size - 1),
                  prob_neighbor_mating=0.9),
            termination,
            seed=seed, save_history=False, verbose=False,
        )
        tempo_moead = perf_counter() - t0

        # CORREÇÃO: o reparo por projeção avalia F no ponto PROJETADO
        # Xp = proj(X), mas o pymoo armazena na população o genótipo X
        # NÃO projetado. Exportar res_moead.X pareava F (do ponto viável
        # projetado) com um X possivelmente FORA da esfera. Exporta-se o
        # X projetado, consistente com o F salvo. Arquivos moead_seed*.csv
        # gerados antes desta correção têm X inviável (F correto).
        X_moead_export = projetar_na_esfera(
            np.asarray(res_moead.X, dtype=float), RAIO_DOE
        )
        caminho_moead = exportar_fronteira_pymoo(
            res_moead.F, X_moead_export, f"moead_seed{seed:03d}.csv"
        )

        linhas_resumo.append({
            "metodo": "MOEA/D", "seed": seed,
            "avaliacoes_rsm": prob_moead.n_avaliacoes_rsm,
            "n_solucoes": len(res_moead.F),
            "tempo_s": tempo_moead,
            "arquivo": str(caminho_moead),
        })

    df_resumo_ea_seeds = pd.DataFrame(linhas_resumo)
    df_resumo_ea_seeds["orcamento_rsm_referencia"] = orcamento_referencia_rsm
    df_resumo_ea_seeds["tempo_parede_metodo_referencia_s"] = tempo_referencia_parede_s
    if np.isfinite(tempo_referencia_parede_s) and tempo_referencia_parede_s > 0:
        df_resumo_ea_seeds["fracao_tempo_metodo"] = (
            df_resumo_ea_seeds["tempo_s"] / tempo_referencia_parede_s
        )
    df_resumo_ea_seeds.to_csv(OUTPUT_DIR / "resumo_ea_mesmo_orcamento_seeds.csv", index=False)

    print("\nResumo por seed salvo em:",
          (OUTPUT_DIR / "resumo_ea_mesmo_orcamento_seeds.csv").resolve())
    print(f"Fronteiras individuais salvas em: {EXPORT_DIR_EAS.resolve()}")

    display(
        df_resumo_ea_seeds
        .groupby("metodo")
        .agg(
            avaliacoes_rsm_media=("avaliacoes_rsm", "mean"),
            avaliacoes_rsm_std=("avaliacoes_rsm", "std"),
            n_solucoes_mediana=("n_solucoes", "median"),
            tempo_s_medio=("tempo_s", "mean"),
        )
        .reset_index()
    )
